In [20]:
# ============================================
# Resolution study (2-session within-subject) analysis pipeline
# UPDATED (per your request):
#   - Loads IAPS valence info from Excel:
#       VALENCE_XLSX = r"C:\Users\pkell\Downloads\Resolution Data\VALARratings_filtered_ordered.xlsx"
#     and uses columns:
#       * "IAPS"  (image ID / key)
#       * "valmn" (numeric IAPS valence)
#   - Matches subject Session1/Session2 CSVs to this valence table via "IAPS" column
#   - Uses that numeric valence for:
#       * Pearson correlation between IAPS valence (valmn) and mean enjoyment ratings (per image)
#       * Deriving valence factor (pos vs neg) for 2×2 RM-ANOVA (VALENCE × RESOLUTION)
#   - Still uses image PATH to determine resolution (high vs low), and pairs sessions within-image
#   - Computes PH/PL/NH/NL participant means + group means/SD
#   - Follow-up paired t-tests within valence (Bonferroni alpha = .025)
#   - RT summaries + optional same 2×2 ANOVA for RT
#   - Plots: rain plot (PH/PL/NH/NL), paired lines (PH vs PL; NH vs NL), and valence correlation scatter
# ============================================
# ============================================

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib qt


# ---------- USER: SET THESE ----------
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Resolution Study\data v1"
FILE_GLOB = "*.csv"     # e.g. "*Resolution Study*.csv"
OUTPUT_DIR = None       # None -> makes "./resolution_analysis_outputs" inside DATA_DIR (or CWD if DATA_DIR blank)

VALENCE_XLSX = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\IAPS Images_before\IAPS Images_before\VALARratings.xlsx"
VALENCE_SHEET = "Sheet1"    # None -> first sheet; OR set to sheet name like "Sheet1"
VALENCE_KEY_COL = "IAPS"
VALENCE_VALMN_COL = "valmn"
# -------------------------------------

# PsychoPy column names
ENJOY_COL = "slider.response"
RT_COL = "slider.rt"
COND1_COL = "cond1"
COND2_COL = "cond2"

# Subject CSV column holding IAPS ID
SUBJ_IAPS_COL = "IAPS"

# Resolution classification keywords in stimulus PATH
HIGH_TOKEN = "stimuli"
LOW_TOKEN = "degraded stimuli"

DROP_MISSING_ENJOY = True
KEEP_DUPLICATE_POLICY = "last"   # "first" or "last"
BONFERRONI_ALPHA = 0.025

# Valence factor settings (IAPS valence is typically 1..9; neutral ~ 5)
VALENCE_NEUTRAL_POINT = 5.0
DROP_NEUTRAL_FOR_ANOVA = True


# -------------------- helpers --------------------
def norm_path(p: str) -> str:
    if pd.isna(p):
        return ""
    p = str(p).strip().replace("\\", "/")
    return p.lower()

def get_pid_from_path(path: str) -> str | None:
    m = re.search(r"(pid\d{3})", path, flags=re.IGNORECASE)
    return m.group(1).lower() if m else None

def get_session_from_path(path: str) -> int | None:
    s = path.lower()
    if "session1" in s or "sess1" in s:
        return 1
    if "session2" in s or "sess2" in s:
        return 2
    return None

def classify_resolution_from_path(stim_path: str) -> str:
    p = norm_path(stim_path)
    if f"/{LOW_TOKEN}/" in p or LOW_TOKEN in p or "degraded" in p:
        return "low"
    if f"/{HIGH_TOKEN}/" in p or HIGH_TOKEN in p:
        return "high"
    return "unknown"

def image_id_from_path(stim_path: str) -> str:
    p = norm_path(stim_path)
    base = os.path.basename(p)
    base = re.sub(r"\.[a-z0-9]+$", "", base)
    return base

def safe_read_csv(fp: str) -> pd.DataFrame:
    return pd.read_csv(fp)

def paired_ttest(x: np.ndarray, y: np.ndarray):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 2:
        return dict(n=n, t=np.nan, p=np.nan, dz=np.nan, mean_x=np.nan, mean_y=np.nan, mean_diff=np.nan)

    d = x - y
    mean_diff = float(np.mean(d))
    sd_diff = float(np.std(d, ddof=1)) if n > 1 else np.nan
    dz = mean_diff / sd_diff if (np.isfinite(sd_diff) and sd_diff != 0) else np.nan

    try:
        from scipy import stats
        t, p = stats.ttest_rel(x, y, nan_policy="omit")
        t, p = float(t), float(p)
    except Exception:
        t, p = np.nan, np.nan

    return dict(n=n, t=t, p=p, dz=dz, mean_diff=mean_diff)

def mean_sd(a: np.ndarray):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan, np.nan, 0
    m = float(np.mean(a))
    sd = float(np.std(a, ddof=1)) if len(a) > 1 else np.nan
    return m, sd, len(a)

def mean_sem(a: np.ndarray):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan, np.nan, 0
    m = float(np.mean(a))
    sem = float(np.std(a, ddof=1) / np.sqrt(len(a))) if len(a) > 1 else np.nan
    return m, sem, len(a)

def valence_factor_from_valmn(valmn: float) -> str:
    if not np.isfinite(valmn):
        return "unknown"
    if valmn > VALENCE_NEUTRAL_POINT:
        return "pos"
    if valmn < VALENCE_NEUTRAL_POINT:
        return "neg"
    return "neutral"

def run_anova_from_cell_means(cell_means_df: pd.DataFrame, dv_col: str, value_col: str):
    try:
        from statsmodels.stats.anova import AnovaRM
        aov = AnovaRM(
            data=cell_means_df.rename(columns={value_col: dv_col}),
            depvar=dv_col,
            subject="pid",
            within=["valence", "resolution"]
        ).fit()
        return aov
    except Exception as e:
        print("\n[ANOVA WARNING] Could not run statsmodels AnovaRM. Error:")
        print(" ", repr(e))
        return None


# -------------------- load IAPS valence table --------------------
print("\nLoading IAPS valence table:", VALENCE_XLSX)
val_df = pd.read_excel(VALENCE_XLSX, sheet_name=VALENCE_SHEET, engine="openpyxl")

if (VALENCE_KEY_COL not in val_df.columns) or (VALENCE_VALMN_COL not in val_df.columns):
    raise KeyError(
        f"Valence file must contain columns {VALENCE_KEY_COL!r} and {VALENCE_VALMN_COL!r}.\n"
        f"Columns present: {list(val_df.columns)}"
    )

val_df["_IAPS_int"] = pd.to_numeric(val_df[VALENCE_KEY_COL], errors="coerce").round()
val_df["_valmn"] = pd.to_numeric(val_df[VALENCE_VALMN_COL], errors="coerce")

val_map = (
    val_df.dropna(subset=["_IAPS_int"])
    .assign(_IAPS_int=val_df["_IAPS_int"].dropna().astype(int))
    .groupby("_IAPS_int", as_index=True)["_valmn"]
    .mean()
    .to_dict()
)

print(f"Valence rows read: {len(val_df)} | Unique IAPS IDs with valmn: {len(val_map)}")


# -------------------- file discovery + pairing --------------------
search_dir = DATA_DIR if DATA_DIR else os.getcwd()
all_files = sorted(glob.glob(os.path.join(search_dir, FILE_GLOB)))

if len(all_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found. Check DATA_DIR and FILE_GLOB.\nDATA_DIR={DATA_DIR!r}, search_dir={search_dir!r}, FILE_GLOB={FILE_GLOB!r}"
    )

pairs = {}  # pid -> {1: fp1, 2: fp2}
unassigned = []

for fp in all_files:
    pid = get_pid_from_path(fp)
    sess = get_session_from_path(fp)
    if (pid is None) or (sess is None):
        unassigned.append(fp)
        continue
    pairs.setdefault(pid, {})
    pairs[pid][sess] = fp

pids = sorted(pairs.keys())
print(f"\nFound {len(all_files)} CSVs total.")
print(f"Identified {len(pids)} participant IDs with at least one session file.")

if unassigned:
    print("\nFiles skipped (couldn't detect pid### and/or session1/session2 in path):")
    for fp in unassigned[:20]:
        print("  -", fp)
    if len(unassigned) > 20:
        print(f"  ... plus {len(unassigned)-20} more")

missing = [(pid, pairs[pid].get(1), pairs[pid].get(2)) for pid in pids if (1 not in pairs[pid] or 2 not in pairs[pid])]
if missing:
    print("\nParticipants missing a session file (will be skipped for paired analyses):")
    for pid, s1, s2 in missing:
        print(f"  {pid}: session1={bool(s1)}, session2={bool(s2)}")


# -------------------- parse each session file into long-form trials --------------------
rows = []

for pid in pids:
    for sess in (1, 2):
        fp = pairs[pid].get(sess, None)
        if not fp:
            continue

        df = safe_read_csv(fp)
        stim_col = COND1_COL if sess == 1 else COND2_COL

        for required in (stim_col, ENJOY_COL, RT_COL, SUBJ_IAPS_COL):
            if required not in df.columns:
                raise KeyError(
                    f"Expected column {required!r} not found in:\n  {fp}\n"
                    f"Columns present: {list(df.columns)}"
                )

        tmp = pd.DataFrame({
            "pid": pid,
            "session": sess,
            "IAPS": pd.to_numeric(df[SUBJ_IAPS_COL], errors="coerce").round(),
            "stim_path": df[stim_col],
            "enjoy_rating": pd.to_numeric(df[ENJOY_COL], errors="coerce"),
            "rt": pd.to_numeric(df[RT_COL], errors="coerce"),
        })

        tmp["stim_path_norm"] = tmp["stim_path"].map(norm_path)
        tmp = tmp[tmp["stim_path_norm"].astype(bool)]

        if DROP_MISSING_ENJOY:
            tmp = tmp[np.isfinite(tmp["enjoy_rating"].to_numpy())]

        tmp = tmp.dropna(subset=["IAPS"]).copy()
        tmp["IAPS"] = tmp["IAPS"].astype(int)

        tmp["iaps_valence"] = tmp["IAPS"].map(val_map).astype(float)
        tmp["valence"] = tmp["iaps_valence"].map(valence_factor_from_valmn)

        tmp["resolution_path"] = tmp["stim_path"].map(classify_resolution_from_path)
        tmp["image_id"] = tmp["stim_path"].map(image_id_from_path)

        if KEEP_DUPLICATE_POLICY in ("first", "last"):
            tmp = tmp.reset_index(drop=False).rename(columns={"index": "_rowidx"})
            tmp = tmp.sort_values("_rowidx", ascending=True)
            tmp = tmp.drop_duplicates(subset=["pid", "session", "IAPS"], keep=KEEP_DUPLICATE_POLICY)
            tmp = tmp.drop(columns=["_rowidx"])

        rows.append(tmp[[
            "pid", "session", "IAPS", "image_id", "stim_path",
            "resolution_path", "enjoy_rating", "rt",
            "iaps_valence", "valence"
        ]])

trials = pd.concat(rows, ignore_index=True)
print(f"\nParsed trials: {len(trials)} rows")
print(trials.head())

# Missing valence report
miss_val = float(trials["iaps_valence"].isna().mean()) if len(trials) else 0.0
if miss_val > 0:
    nmiss = int(trials["iaps_valence"].isna().sum())
    print(f"\nWARNING: {nmiss} trials ({miss_val*100:.2f}%) have missing iaps_valence (valmn) from the Excel lookup.")


# -------------------- within-participant pairing by IAPS across sessions --------------------
paired_pids = [pid for pid in pids if (1 in pairs.get(pid, {}) and 2 in pairs.get(pid, {}))]
if len(paired_pids) == 0:
    raise RuntimeError("No participants with both session1 and session2 detected.")

t1 = trials[trials["session"] == 1].rename(columns={
    "stim_path": "stim_s1",
    "resolution_path": "res_s1",
    "enjoy_rating": "enjoy_s1",
    "rt": "rt_s1",
    "iaps_valence": "iaps_valence_s1",
    "valence": "valence_s1",
})[["pid", "IAPS", "image_id", "stim_s1", "res_s1", "enjoy_s1", "rt_s1", "iaps_valence_s1", "valence_s1"]]

t2 = trials[trials["session"] == 2].rename(columns={
    "stim_path": "stim_s2",
    "resolution_path": "res_s2",
    "enjoy_rating": "enjoy_s2",
    "rt": "rt_s2",
    "iaps_valence": "iaps_valence_s2",
    "valence": "valence_s2",
})[["pid", "IAPS", "image_id", "stim_s2", "res_s2", "enjoy_s2", "rt_s2", "iaps_valence_s2", "valence_s2"]]

paired = pd.merge(t1, t2, on=["pid", "IAPS"], how="inner")
paired = paired[paired["pid"].isin(paired_pids)].reset_index(drop=True)
print(f"\nPaired images across sessions: {len(paired)} rows (across {paired['pid'].nunique()} participants)")

paired["iaps_valence"] = paired[["iaps_valence_s1", "iaps_valence_s2"]].mean(axis=1, skipna=True)
paired["valence"] = np.where(paired["valence_s1"] == paired["valence_s2"], paired["valence_s1"], paired["valence_s1"])

def pick_high_low(row):
    enjoy_high = np.nan
    enjoy_low = np.nan
    rt_high = np.nan
    rt_low = np.nan

    if row["res_s1"] == "high":
        enjoy_high, rt_high = row["enjoy_s1"], row["rt_s1"]
    if row["res_s2"] == "high":
        enjoy_high, rt_high = row["enjoy_s2"], row["rt_s2"]

    if row["res_s1"] == "low":
        enjoy_low, rt_low = row["enjoy_s1"], row["rt_s1"]
    if row["res_s2"] == "low":
        enjoy_low, rt_low = row["enjoy_s2"], row["rt_s2"]

    return pd.Series({
        "enjoy_high": enjoy_high,
        "enjoy_low": enjoy_low,
        "rt_high": rt_high,
        "rt_low": rt_low,
        "enjoy_high_minus_low": (enjoy_high - enjoy_low) if np.isfinite(enjoy_high) and np.isfinite(enjoy_low) else np.nan,
        "rt_low_minus_high": (rt_low - rt_high) if np.isfinite(rt_low) and np.isfinite(rt_high) else np.nan,
        "enjoy_s2_minus_s1": (row["enjoy_s2"] - row["enjoy_s1"]) if np.isfinite(row["enjoy_s2"]) and np.isfinite(row["enjoy_s1"]) else np.nan,
        "rt_s2_minus_s1": (row["rt_s2"] - row["rt_s1"]) if np.isfinite(row["rt_s2"]) and np.isfinite(row["rt_s1"]) else np.nan,
    })

paired = pd.concat([paired, paired.apply(pick_high_low, axis=1)], axis=1)

paired["valence"] = paired["valence"].fillna("unknown")

# -------------------- build long-form for 2×2 ANOVA --------------------
anov_long = pd.concat([
    paired[["pid", "IAPS", "valence", "iaps_valence", "enjoy_high"]].rename(columns={"enjoy_high": "enjoy"}).assign(resolution="high"),
    paired[["pid", "IAPS", "valence", "iaps_valence", "enjoy_low"]].rename(columns={"enjoy_low": "enjoy"}).assign(resolution="low"),
], ignore_index=True)
anov_long = anov_long[np.isfinite(anov_long["enjoy"].to_numpy())]
anov_long = anov_long[anov_long["resolution"].isin(["high", "low"])]

if DROP_NEUTRAL_FOR_ANOVA:
    anov_long = anov_long[anov_long["valence"].isin(["pos", "neg"])]

anov_long_rt = pd.concat([
    paired[["pid", "IAPS", "valence", "iaps_valence", "rt_high"]].rename(columns={"rt_high": "rt"}).assign(resolution="high"),
    paired[["pid", "IAPS", "valence", "iaps_valence", "rt_low"]].rename(columns={"rt_low": "rt"}).assign(resolution="low"),
], ignore_index=True)
anov_long_rt = anov_long_rt[np.isfinite(anov_long_rt["rt"].to_numpy())]
anov_long_rt = anov_long_rt[anov_long_rt["resolution"].isin(["high", "low"])]

if DROP_NEUTRAL_FOR_ANOVA:
    anov_long_rt = anov_long_rt[anov_long_rt["valence"].isin(["pos", "neg"])]

cell_means = (
    anov_long.groupby(["pid", "valence", "resolution"], as_index=False)
    .agg(mean_enjoy=("enjoy", "mean"))
)
cell_means_rt = (
    anov_long_rt.groupby(["pid", "valence", "resolution"], as_index=False)
    .agg(mean_rt=("rt", "mean"))
)

cell_wide = cell_means.pivot(index="pid", columns=["valence", "resolution"], values="mean_enjoy")
cell_wide.columns = [f"{v.upper()}{'H' if r=='high' else 'L'}" for v, r in cell_wide.columns]
cell_wide = cell_wide.reset_index().rename(columns={"POSH": "PH", "POSL": "PL", "NEGH": "NH", "NEGL": "NL"})

cell_wide_rt = cell_means_rt.pivot(index="pid", columns=["valence", "resolution"], values="mean_rt")
cell_wide_rt.columns = [f"{v.upper()}{'H' if r=='high' else 'L'}" for v, r in cell_wide_rt.columns]
cell_wide_rt = cell_wide_rt.reset_index().rename(columns={"POSH": "PH", "POSL": "PL", "NEGH": "NH", "NEGL": "NL"})

print("\nParticipant-level PH/PL/NH/NL (enjoy) preview:")
print(cell_wide.head())

# -------------------- descriptives --------------------
def report_cell(label, x):
    m, sd, n = mean_sd(x)
    return dict(cell=label, mean=m, sd=sd, n=n)

group_cells = []
for lab in ["PH", "PL", "NH", "NL"]:
    if lab in cell_wide.columns:
        group_cells.append(report_cell(lab, cell_wide[lab].to_numpy()))
group_cells_df = pd.DataFrame(group_cells)

print("\n===== Group descriptives (per-subject means) =====")
print(group_cells_df.to_string(index=False, float_format=lambda z: f"{z:.3f}"))

# -------------------- ANOVA --------------------
aov_enjoy = run_anova_from_cell_means(cell_means, dv_col="enjoy", value_col="mean_enjoy")
aov_rt = run_anova_from_cell_means(cell_means_rt, dv_col="rt", value_col="mean_rt")

print("\n===== 2×2 RM-ANOVA (Enjoy): valence(pos/neg) × resolution(high/low) =====")
print(aov_enjoy if aov_enjoy is not None else "ANOVA not available (see warning above).")

print("\n===== 2×2 RM-ANOVA (RT): valence(pos/neg) × resolution(high/low) =====")
print(aov_rt if aov_rt is not None else "ANOVA not available (see warning above).")

# -------------------- follow-up paired t-tests --------------------
pos = cell_wide.dropna(subset=["PH", "PL"], how="any")
neg = cell_wide.dropna(subset=["NH", "NL"], how="any")

tt_pos = paired_ttest(pos["PH"].to_numpy(), pos["PL"].to_numpy())
tt_neg = paired_ttest(neg["NH"].to_numpy(), neg["NL"].to_numpy())

print("\n===== Follow-up paired t-tests (Bonferroni alpha=.025) =====")
print(f"Positive (PH vs PL): t({tt_pos['n']-1})={tt_pos['t']:.3f}, p={tt_pos['p']}, dz={tt_pos['dz']:.3f}, mean(PH-PL)={tt_pos['mean_diff']:.3f} "
      f"=> {'SIGNIFICANT' if (np.isfinite(tt_pos['p']) and tt_pos['p'] < BONFERRONI_ALPHA) else 'n.s.'}")
print(f"Negative (NH vs NL): t({tt_neg['n']-1})={tt_neg['t']:.3f}, p={tt_neg['p']}, dz={tt_neg['dz']:.3f}, mean(NH-NL)={tt_neg['mean_diff']:.3f} "
      f"=> {'SIGNIFICANT' if (np.isfinite(tt_neg['p']) and tt_neg['p'] < BONFERRONI_ALPHA) else 'n.s.'}")

# -------------------- session-level RT summaries --------------------
subj_session = (
    paired.groupby("pid", as_index=False)
    .agg(
        n_images=("IAPS", "count"),
        mean_enjoy_s1=("enjoy_s1", "mean"),
        mean_enjoy_s2=("enjoy_s2", "mean"),
        mean_rt_s1=("rt_s1", "mean"),
        mean_rt_s2=("rt_s2", "mean"),
        mean_rt_s2_minus_s1=("rt_s2_minus_s1", "mean"),
    )
)
print("\nParticipant-level session summary preview:")
print(subj_session.head())

# -------------------- IAPS valence correlation --------------------
corr_info = None
if np.isfinite(paired["iaps_valence"].to_numpy()).sum() >= 10:
    per_image = (
        pd.concat([
            paired[["IAPS", "iaps_valence", "enjoy_s1"]].rename(columns={"enjoy_s1": "enjoy"}),
            paired[["IAPS", "iaps_valence", "enjoy_s2"]].rename(columns={"enjoy_s2": "enjoy"}),
        ], ignore_index=True)
        .dropna(subset=["iaps_valence", "enjoy"])
        .groupby(["IAPS", "iaps_valence"], as_index=False)
        .agg(mean_enjoy=("enjoy", "mean"))
    )

    try:
        from scipy import stats
        r, p = stats.pearsonr(per_image["iaps_valence"].to_numpy(), per_image["mean_enjoy"].to_numpy())
        corr_info = dict(r=float(r), p=float(p), n=len(per_image))
        print("\n===== IAPS valence correlation (valmn vs mean enjoyment per image) =====")
        print(f"Pearson r = {corr_info['r']:.3f}, p = {corr_info['p']}, N_images = {corr_info['n']}")
    except Exception as e:
        print("\n[CORRELATION WARNING] Could not compute Pearson correlation. Error:", repr(e))
        per_image = None
else:
    per_image = None
    print("\nNOTE: Too few numeric iaps_valence values after Excel merge to run correlation.")

# -------------------- output directory + saves --------------------
if OUTPUT_DIR is None:
    base = search_dir if DATA_DIR else os.getcwd()
    OUTPUT_DIR = os.path.join(base, "resolution_analysis_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

trials_out = os.path.join(OUTPUT_DIR, "trials_long.csv")
paired_out = os.path.join(OUTPUT_DIR, "paired_by_IAPS.csv")
cell_enjoy_out = os.path.join(OUTPUT_DIR, "subject_cellmeans_enjoy_PH_PL_NH_NL.csv")
cell_rt_out = os.path.join(OUTPUT_DIR, "subject_cellmeans_rt_PH_PL_NH_NL.csv")
session_out = os.path.join(OUTPUT_DIR, "subject_session_summary.csv")
group_cells_out = os.path.join(OUTPUT_DIR, "group_cell_descriptives.csv")

trials.to_csv(trials_out, index=False)
paired.to_csv(paired_out, index=False)
cell_wide.to_csv(cell_enjoy_out, index=False)
cell_wide_rt.to_csv(cell_rt_out, index=False)
subj_session.to_csv(session_out, index=False)
group_cells_df.to_csv(group_cells_out, index=False)

print(f"\nSaved outputs to: {OUTPUT_DIR}")
print("  -", trials_out)
print("  -", paired_out)
print("  -", cell_enjoy_out)
print("  -", cell_rt_out)
print("  -", session_out)
print("  -", group_cells_out)

# -------------------- plotting helpers --------------------
def paired_lines_plot(values_a, values_b, label_a, label_b, ylabel, title, save_name):
    a = np.asarray(values_a, dtype=float)
    b = np.asarray(values_b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]

    fig, ax = plt.subplots(figsize=(7, 5))
    for i in range(len(a)):
        ax.plot([0, 1], [a[i], b[i]], marker="o")
    ax.set_xticks([0, 1])
    ax.set_xticklabels([label_a, label_b])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()

def bar_sem_plot(means, sems, labels, ylabel, title, save_name):
    fig, ax = plt.subplots(figsize=(7, 5))
    x = np.arange(len(means))
    ax.bar(x, means, yerr=sems, capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()

def rain_plot(values_by_label, labels, ylabel, title, save_name):
    fig, ax = plt.subplots(figsize=(9, 5))
    data = [np.asarray(v, dtype=float) for v in values_by_label]
    data = [v[np.isfinite(v)] for v in data]

    ax.violinplot(data, showmeans=False, showmedians=True, showextrema=False)

    rng = np.random.default_rng(0)
    for i, v in enumerate(data, start=1):
        jitter = rng.normal(loc=0.0, scale=0.06, size=len(v))
        ax.scatter(np.full_like(v, i, dtype=float) + jitter, v, s=20, alpha=0.8)

    ax.set_xticks(np.arange(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()

# -------------------- plots: PH/PL/NH/NL enjoy --------------------
if all(c in cell_wide.columns for c in ["PH", "PL", "NH", "NL"]):
    m_PH, sem_PH, _ = mean_sem(cell_wide["PH"].to_numpy())
    m_PL, sem_PL, _ = mean_sem(cell_wide["PL"].to_numpy())
    m_NH, sem_NH, _ = mean_sem(cell_wide["NH"].to_numpy())
    m_NL, sem_NL, _ = mean_sem(cell_wide["NL"].to_numpy())

    rain_plot(
        values_by_label=[cell_wide["PH"].to_numpy(), cell_wide["PL"].to_numpy(),
                         cell_wide["NH"].to_numpy(), cell_wide["NL"].to_numpy()],
        labels=["PH", "PL", "NH", "NL"],
        ylabel="Mean enjoyment rating (per subject)",
        title="Rain plot: subject mean ratings by condition (PH/PL/NH/NL)",
        save_name="rain_PH_PL_NH_NL_enjoy.png"
    )

    bar_sem_plot(
        means=[m_PH, m_PL, m_NH, m_NL],
        sems=[sem_PH, sem_PL, sem_NH, sem_NL],
        labels=["PH", "PL", "NH", "NL"],
        ylabel="Mean enjoyment rating (per subject)",
        title="Group mean ratings by condition (PH/PL/NH/NL)",
        save_name="group_bar_PH_PL_NH_NL_enjoy.png"
    )

# paired lines within valence
if all(c in cell_wide.columns for c in ["PH", "PL"]):
    paired_lines_plot(cell_wide["PH"], cell_wide["PL"],
                      "PH (pos high-res)", "PL (pos low-res)",
                      ylabel="Mean enjoyment rating",
                      title="Positive images: PH vs PL (within-subject)",
                      save_name="paired_PH_vs_PL.png")

if all(c in cell_wide.columns for c in ["NH", "NL"]):
    paired_lines_plot(cell_wide["NH"], cell_wide["NL"],
                      "NH (neg high-res)", "NL (neg low-res)",
                      ylabel="Mean enjoyment rating",
                      title="Negative images: NH vs NL (within-subject)",
                      save_name="paired_NH_vs_NL.png")

# IAPS valence correlation plot
if per_image is not None and corr_info is not None:
    x = per_image["iaps_valence"].to_numpy(dtype=float)
    y = per_image["mean_enjoy"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(x, y, s=25, alpha=0.85)
    try:
        coeff = np.polyfit(x, y, deg=1)
        xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
        yy = coeff[0] * xx + coeff[1]
        ax.plot(xx, yy)
    except Exception:
        pass

    ax.set_xlabel("IAPS valence (valmn)")
    ax.set_ylabel("Mean participant enjoyment rating (per image)")
    ax.set_title(f"IAPS valence vs mean enjoyment: r={corr_info['r']:.3f}, p={corr_info['p']}")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "iaps_valence_correlation.png"), dpi=150)
    plt.show()

print("\nAll plots saved to:", OUTPUT_DIR)



Loading IAPS valence table: C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\IAPS Images_before\IAPS Images_before\VALARratings.xlsx
Valence rows read: 1194 | Unique IAPS IDs with valmn: 1164

Found 44 CSVs total.
Identified 22 participant IDs with at least one session file.

Parsed trials: 14565 rows
      pid  session  IAPS image_id  \
0  pid001        1  2345     2345   
1  pid001        1  2071     2071   
2  pid001        1  9160     9160   
3  pid001        1  2100     2100   
4  pid001        1  8030     8030   

                                           stim_path resolution_path  \
0  /Users/shamslab/Downloads/Resolution Study/sti...            high   
1  /Users/shamslab/Downloads/Resolution Study/sti...            high   
2  /Users/shamslab/Downloads/Resolution Study/sti...            high   
3  /Users/shamslab/Downloads/Resolution Study/sti...             low   
4  /Users/shamslab/Downloads/Resolution Study/sti...             low   

   enjoy_rating        rt  iaps_v

In [22]:
# =====================================================================
# NEGATIVE-IMAGE NULL / EQUIVALENCE ANALYSIS
# =====================================================================
#
# PURPOSE
# -------
# Tests whether the high-vs-low resolution difference for NEGATIVE
# images is small/negligible rather than merely nonsignificant.
#
# Uses the SAME participant-level NH and NL means used by the
# repeated-measures ANOVA and paired t test.
#
# Analyses:
#   1. Standard paired-samples t test
#   2. 95% CI for NH - NL
#   3. TOST equivalence tests at:
#          dz = ±0.20
#          dz = ±0.30   <-- recommended primary SESOI
#          dz = ±0.50
#   4. JZS Bayesian paired t test
#   5. Bayes-factor prior-width sensitivity analysis
#   6. Saves all results to CSV
#
# IMPORTANT
# ---------
# Run this cell AFTER the main ANOVA cell that creates `cell_wide`.
#
# This version deliberately DOES NOT use `paired`, because another
# analysis cell in the notebook overwrites `paired` with a different
# dataframe schema.
# =====================================================================


# ---------------------------------------------------------------------
# 1. IMPORTS
# ---------------------------------------------------------------------

import os
import numpy as np
import pandas as pd

from scipy import stats
from scipy.integrate import quad


# =====================================================================
# 2. USER SETTINGS
# =====================================================================

EXPERIMENT_LABEL = "Experiment 1"

# Standard alpha for TOST
TOST_ALPHA = 0.05

# Standardized equivalence bounds to evaluate
#
# dz ±0.30 is the recommended primary SESOI.
# ±0.20 and ±0.50 are included as sensitivity analyses.
SESOI_DZ_VALUES = [
    0.20,
    0.30,
    0.50,
]


# Primary JZS Cauchy prior width
JZS_R_PRIMARY = np.sqrt(2) / 2     # ≈ 0.707


# Prior-width sensitivity analysis
JZS_R_VALUES = [
    0.50,
    np.sqrt(2) / 2,
    1.00,
]


# Output folder name
OUTPUT_SUBFOLDER = "negative_visual_null_evidence"


# =====================================================================
# 3. GET THE PARTICIPANT-LEVEL NH / NL DATA
# =====================================================================
#
# Preferred source:
#     `cell_wide`
#
# The main ANOVA pipeline creates:
#
#     pid | NH | NL | PH | PL
#
# These are exactly the participant-level condition means used for
# the ANOVA and follow-up paired t tests.
#
# If `cell_wide` is unavailable, this code attempts to load the saved
# CSV created by the main pipeline.
# =====================================================================


def locate_cell_wide():
    """
    Return a participant-level dataframe containing:
        pid, NH, NL, PH, PL

    Priority:
        1. Existing `cell_wide` object in notebook memory
        2. Saved subject_cellmeans_enjoy_PH_PL_NH_NL.csv
    """

    # -------------------------------------------------------------
    # OPTION 1:
    # Use cell_wide already in notebook memory
    # -------------------------------------------------------------

    if "cell_wide" in globals():

        candidate = globals()["cell_wide"]

        if isinstance(candidate, pd.DataFrame):

            required = {"pid", "NH", "NL"}

            if required.issubset(candidate.columns):

                print(
                    "Using participant-level `cell_wide` "
                    "already present in notebook memory."
                )

                return candidate.copy()


    # -------------------------------------------------------------
    # OPTION 2:
    # Load the CSV produced by the main ANOVA pipeline
    # -------------------------------------------------------------

    candidate_paths = []


    # If OUTPUT_DIR exists and is a valid string
    if (
        "OUTPUT_DIR" in globals()
        and isinstance(globals()["OUTPUT_DIR"], str)
        and globals()["OUTPUT_DIR"].strip()
    ):

        candidate_paths.append(
            os.path.join(
                globals()["OUTPUT_DIR"],
                "subject_cellmeans_enjoy_PH_PL_NH_NL.csv",
            )
        )


    # If DATA_DIR exists, reproduce the notebook's default output path
    if (
        "DATA_DIR" in globals()
        and isinstance(globals()["DATA_DIR"], str)
        and globals()["DATA_DIR"].strip()
    ):

        candidate_paths.append(
            os.path.join(
                globals()["DATA_DIR"],
                "resolution_analysis_outputs",
                "subject_cellmeans_enjoy_PH_PL_NH_NL.csv",
            )
        )


    # Remove duplicate paths
    candidate_paths = list(
        dict.fromkeys(candidate_paths)
    )


    for path in candidate_paths:

        if os.path.exists(path):

            tmp = pd.read_csv(path)

            required = {"pid", "NH", "NL"}

            if required.issubset(tmp.columns):

                print(
                    "Loaded participant-level condition means from:"
                )

                print(path)

                return tmp.copy()


    # -------------------------------------------------------------
    # Nothing usable was found
    # -------------------------------------------------------------

    raise RuntimeError(
        "\nCould not locate a valid participant-level "
        "`cell_wide` dataframe.\n\n"
        "Please run the main Resolution Study ANOVA cell first. "
        "That cell creates participant-level PH, PL, NH, and NL "
        "condition means.\n\n"
        "Required columns:\n"
        "    pid, NH, NL\n"
    )


analysis_df = locate_cell_wide()


# =====================================================================
# 4. CLEAN THE NEGATIVE-CONDITION DATA
# =====================================================================

negative_df = analysis_df[
    [
        "pid",
        "NH",
        "NL",
    ]
].copy()


# Convert ratings to numeric
negative_df["NH"] = pd.to_numeric(
    negative_df["NH"],
    errors="coerce",
)

negative_df["NL"] = pd.to_numeric(
    negative_df["NL"],
    errors="coerce",
)


# Keep participants with BOTH conditions
negative_df = (
    negative_df
    .dropna(
        subset=[
            "NH",
            "NL",
        ]
    )
    .copy()
)


# Paired difference:
#
# NH - NL
#
# positive:
#     high-resolution negative images were rated more positively
#
# negative:
#     high-resolution negative images were rated more negatively
#
negative_df["NH_minus_NL"] = (
    negative_df["NH"]
    - negative_df["NL"]
)


# =====================================================================
# 5. BASIC DATA CHECKS
# =====================================================================

n = len(negative_df)

if n < 2:
    raise RuntimeError(
        "Fewer than two participants have usable NH and NL values."
    )


NH = negative_df[
    "NH"
].to_numpy(
    dtype=float
)

NL = negative_df[
    "NL"
].to_numpy(
    dtype=float
)

diff = negative_df[
    "NH_minus_NL"
].to_numpy(
    dtype=float
)


df = n - 1


print("\n" + "=" * 80)
print("NEGATIVE-IMAGE NULL / EQUIVALENCE ANALYSIS")
print("=" * 80)

print(
    f"\nExperiment: {EXPERIMENT_LABEL}"
)

print(
    f"Participants included: {n}"
)

print(
    f"Participant IDs:\n"
    f"{negative_df['pid'].tolist()}"
)


# =====================================================================
# 6. STANDARD PAIRED-EFFECT STATISTICS
# =====================================================================

mean_NH = float(
    np.mean(NH)
)

mean_NL = float(
    np.mean(NL)
)

mean_diff = float(
    np.mean(diff)
)

sd_diff = float(
    np.std(
        diff,
        ddof=1,
    )
)

se_diff = float(
    sd_diff
    / np.sqrt(n)
)


if (
    not np.isfinite(sd_diff)
    or sd_diff <= 0
):

    raise RuntimeError(
        "The SD of NH - NL difference scores "
        "is zero or invalid."
    )


# ---------------------------------------------------------------------
# Paired-samples t test
# ---------------------------------------------------------------------

t_stat, p_two_sided = stats.ttest_rel(
    NH,
    NL,
)

t_stat = float(
    t_stat
)

p_two_sided = float(
    p_two_sided
)


# Cohen's dz
cohens_dz = float(
    mean_diff
    / sd_diff
)


# ---------------------------------------------------------------------
# 95% confidence interval for NH - NL
# ---------------------------------------------------------------------

tcrit_95 = stats.t.ppf(
    0.975,
    df,
)

ci95_low = float(
    mean_diff
    - tcrit_95 * se_diff
)

ci95_high = float(
    mean_diff
    + tcrit_95 * se_diff
)


print("\n" + "-" * 80)
print("STANDARD PAIRED COMPARISON")
print("-" * 80)

print(
    f"\nNH mean = {mean_NH:.3f}"
)

print(
    f"NL mean = {mean_NL:.3f}"
)

print(
    f"NH - NL mean difference = "
    f"{mean_diff:.3f}"
)

print(
    f"SD of paired differences = "
    f"{sd_diff:.3f}"
)

print(
    f"SE of paired difference = "
    f"{se_diff:.3f}"
)

print(
    f"t({df}) = "
    f"{t_stat:.3f}"
)

print(
    f"p = "
    f"{p_two_sided:.6f}"
)

print(
    f"Cohen's dz = "
    f"{cohens_dz:.3f}"
)

print(
    f"95% CI for NH - NL = "
    f"[{ci95_low:.3f}, "
    f"{ci95_high:.3f}]"
)


# =====================================================================
# 7. TOST EQUIVALENCE TEST FUNCTION
# =====================================================================
#
# Tests whether:
#
#      -bound < NH - NL < +bound
#
# The standardized dz bound is converted into raw enjoyment units
# using the observed SD of the within-participant difference scores.
#
# =====================================================================


def run_tost_dz(
    difference_scores,
    sesoi_dz,
    alpha=0.05,
):

    difference_scores = np.asarray(
        difference_scores,
        dtype=float,
    )

    difference_scores = difference_scores[
        np.isfinite(
            difference_scores
        )
    ]

    n_local = len(
        difference_scores
    )

    df_local = (
        n_local - 1
    )

    mean_local = float(
        np.mean(
            difference_scores
        )
    )

    sd_local = float(
        np.std(
            difference_scores,
            ddof=1,
        )
    )

    se_local = float(
        sd_local
        / np.sqrt(
            n_local
        )
    )


    # -------------------------------------------------------------
    # Convert dz SESOI to raw-score equivalence bound
    # -------------------------------------------------------------

    raw_bound = float(
        abs(sesoi_dz)
        * sd_local
    )


    lower_bound = (
        -raw_bound
    )

    upper_bound = (
        raw_bound
    )


    # -------------------------------------------------------------
    # Lower one-sided test
    #
    # H0: true effect <= lower bound
    # H1: true effect > lower bound
    # -------------------------------------------------------------

    t_lower = float(
        (
            mean_local
            - lower_bound
        )
        / se_local
    )

    p_lower = float(
        stats.t.sf(
            t_lower,
            df_local,
        )
    )


    # -------------------------------------------------------------
    # Upper one-sided test
    #
    # H0: true effect >= upper bound
    # H1: true effect < upper bound
    # -------------------------------------------------------------

    t_upper = float(
        (
            mean_local
            - upper_bound
        )
        / se_local
    )

    p_upper = float(
        stats.t.cdf(
            t_upper,
            df_local,
        )
    )


    # Overall TOST p-value
    p_tost = float(
        max(
            p_lower,
            p_upper,
        )
    )


    equivalent = bool(
        (
            p_lower < alpha
        )
        and
        (
            p_upper < alpha
        )
    )


    # -------------------------------------------------------------
    # TOST confidence interval
    #
    # alpha=.05 -> 90% CI
    # -------------------------------------------------------------

    ci_level = (
        1.0
        - 2.0 * alpha
    )

    tcrit = stats.t.ppf(
        1.0 - alpha,
        df_local,
    )

    ci_low = float(
        mean_local
        - tcrit * se_local
    )

    ci_high = float(
        mean_local
        + tcrit * se_local
    )


    return {
        "SESOI_dz":
            float(sesoi_dz),

        "raw_lower_bound":
            float(lower_bound),

        "raw_upper_bound":
            float(upper_bound),

        "t_lower":
            t_lower,

        "p_lower":
            p_lower,

        "t_upper":
            t_upper,

        "p_upper":
            p_upper,

        "TOST_p":
            p_tost,

        "CI_level":
            ci_level,

        "CI_low":
            ci_low,

        "CI_high":
            ci_high,

        "equivalent":
            equivalent,
    }


# =====================================================================
# 8. RUN TOST AT dz = .20, .30, .50
# =====================================================================

tost_results = []


for sesoi in SESOI_DZ_VALUES:

    result = run_tost_dz(
        difference_scores=diff,
        sesoi_dz=sesoi,
        alpha=TOST_ALPHA,
    )

    tost_results.append(
        result
    )


tost_df = pd.DataFrame(
    tost_results
)


print("\n" + "=" * 80)
print("TOST EQUIVALENCE TESTS")
print("=" * 80)


for _, row in tost_df.iterrows():

    print(
        f"\nSESOI: dz = ±{row['SESOI_dz']:.2f}"
    )

    print(
        f"Raw equivalence bounds = "
        f"[{row['raw_lower_bound']:.3f}, "
        f"{row['raw_upper_bound']:.3f}]"
    )

    print(
        f"Lower test: "
        f"t({df}) = {row['t_lower']:.3f}, "
        f"p = {row['p_lower']:.6f}"
    )

    print(
        f"Upper test: "
        f"t({df}) = {row['t_upper']:.3f}, "
        f"p = {row['p_upper']:.6f}"
    )

    print(
        f"TOST p = "
        f"{row['TOST_p']:.6f}"
    )

    print(
        f"{row['CI_level'] * 100:.0f}% CI = "
        f"[{row['CI_low']:.3f}, "
        f"{row['CI_high']:.3f}]"
    )

    print(
        "Conclusion: "
        + (
            "EQUIVALENT"
            if row["equivalent"]
            else "EQUIVALENCE NOT ESTABLISHED"
        )
    )


# =====================================================================
# 9. JZS BAYES FACTOR
# =====================================================================
#
# BF10:
#     evidence for a nonzero effect relative to H0
#
# BF01:
#     evidence for H0 relative to a nonzero-effect alternative
#
# Uses a two-sided JZS prior.
#
# =====================================================================


def jzs_bf10_ttest(
    t_value,
    n_pairs,
    r=np.sqrt(2) / 2,
):

    t_value = float(
        t_value
    )

    n_pairs = int(
        n_pairs
    )

    r = float(
        r
    )

    if n_pairs < 2:
        raise ValueError(
            "n_pairs must be >= 2."
        )

    if r <= 0:
        raise ValueError(
            "r must be > 0."
        )


    v = (
        n_pairs - 1
    )

    N = float(
        n_pairs
    )


    # -------------------------------------------------------------
    # JZS marginal likelihood integral
    # -------------------------------------------------------------

    def integrand(g):

        return (

            (
                1.0
                + N * g * r**2
            )
            ** (-0.5)

            *

            (
                1.0
                +
                (
                    t_value**2
                    /
                    (
                        v
                        *
                        (
                            1.0
                            + N * g * r**2
                        )
                    )
                )
            )
            ** (
                -(v + 1.0)
                / 2.0
            )

            *

            (
                2.0
                * np.pi
            )
            ** (-0.5)

            *

            g ** (-1.5)

            *

            np.exp(
                -1.0
                /
                (
                    2.0 * g
                )
            )
        )


    numerator, error = quad(
        integrand,
        0.0,
        np.inf,
        limit=500,
    )


    denominator = (
        1.0
        +
        (
            t_value**2
            / v
        )
    ) ** (
        -(v + 1.0)
        / 2.0
    )


    BF10 = float(
        numerator
        / denominator
    )


    return BF10


# =====================================================================
# 10. BAYES-FACTOR PRIOR SENSITIVITY
# =====================================================================

bf_results = []


for r_value in JZS_R_VALUES:

    BF10 = jzs_bf10_ttest(
        t_value=t_stat,
        n_pairs=n,
        r=r_value,
    )

    BF01 = float(
        1.0 / BF10
    )

    bf_results.append(
        {
            "Cauchy_r":
                float(r_value),

            "BF10":
                BF10,

            "BF01":
                BF01,
        }
    )


bf_df = pd.DataFrame(
    bf_results
)


# Primary row
primary_idx = np.argmin(
    np.abs(
        bf_df["Cauchy_r"].to_numpy()
        - JZS_R_PRIMARY
    )
)

BF10_primary = float(
    bf_df.iloc[
        primary_idx
    ]["BF10"]
)

BF01_primary = float(
    bf_df.iloc[
        primary_idx
    ]["BF01"]
)


# =====================================================================
# 11. BAYES-FACTOR INTERPRETATION
# =====================================================================


def interpret_bf01(
    bf01,
):

    if bf01 < 1:

        return (
            "evidence favors a nonzero effect"
        )

    elif bf01 < 3:

        return (
            "anecdotal/weak evidence for H0"
        )

    elif bf01 < 10:

        return (
            "moderate evidence for H0"
        )

    elif bf01 < 30:

        return (
            "strong evidence for H0"
        )

    elif bf01 < 100:

        return (
            "very strong evidence for H0"
        )

    else:

        return (
            "extreme evidence for H0"
        )


print("\n" + "=" * 80)
print("JZS BAYES FACTOR")
print("=" * 80)


print(
    f"\nPrimary prior width r = "
    f"{JZS_R_PRIMARY:.3f}"
)

print(
    f"BF10 = "
    f"{BF10_primary:.4f}"
)

print(
    f"BF01 = "
    f"{BF01_primary:.4f}"
)

print(
    f"Interpretation: "
    f"{interpret_bf01(BF01_primary)}"
)


print(
    "\nPrior-width sensitivity:"
)

print(
    bf_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# =====================================================================
# 12. CREATE OVERALL SUMMARY TABLE
# =====================================================================

summary_df = pd.DataFrame(
    [
        {
            "experiment":
                EXPERIMENT_LABEL,

            "n":
                n,

            "mean_NH":
                mean_NH,

            "mean_NL":
                mean_NL,

            "mean_NH_minus_NL":
                mean_diff,

            "sd_difference":
                sd_diff,

            "se_difference":
                se_diff,

            "t":
                t_stat,

            "df":
                df,

            "p":
                p_two_sided,

            "cohens_dz":
                cohens_dz,

            "CI95_low":
                ci95_low,

            "CI95_high":
                ci95_high,

            "JZS_primary_r":
                JZS_R_PRIMARY,

            "BF10":
                BF10_primary,

            "BF01":
                BF01_primary,
        }
    ]
)


# =====================================================================
# 13. DETERMINE OUTPUT DIRECTORY
# =====================================================================

# Use existing OUTPUT_DIR if valid
if (
    "OUTPUT_DIR" in globals()
    and isinstance(
        globals()["OUTPUT_DIR"],
        str,
    )
    and globals()["OUTPUT_DIR"].strip()
):

    base_output_dir = globals()[
        "OUTPUT_DIR"
    ]


# Otherwise reproduce default notebook output directory
elif (
    "DATA_DIR" in globals()
    and isinstance(
        globals()["DATA_DIR"],
        str,
    )
    and globals()["DATA_DIR"].strip()
):

    base_output_dir = os.path.join(
        globals()["DATA_DIR"],
        "resolution_analysis_outputs",
    )


# Final fallback
else:

    base_output_dir = os.path.join(
        os.getcwd(),
        "resolution_analysis_outputs",
    )


null_output_dir = os.path.join(
    base_output_dir,
    OUTPUT_SUBFOLDER,
)


os.makedirs(
    null_output_dir,
    exist_ok=True,
)


# =====================================================================
# 14. SAVE RESULTS
# =====================================================================

participant_path = os.path.join(
    null_output_dir,
    "negative_condition_participant_means.csv",
)

summary_path = os.path.join(
    null_output_dir,
    "negative_condition_standard_and_bayes_summary.csv",
)

tost_path = os.path.join(
    null_output_dir,
    "negative_condition_TOST_sensitivity.csv",
)

bf_path = os.path.join(
    null_output_dir,
    "negative_condition_BayesFactor_sensitivity.csv",
)


negative_df.to_csv(
    participant_path,
    index=False,
)

summary_df.to_csv(
    summary_path,
    index=False,
)

tost_df.to_csv(
    tost_path,
    index=False,
)

bf_df.to_csv(
    bf_path,
    index=False,
)


# =====================================================================
# 15. FINAL MANUSCRIPT-RELEVANT OUTPUT
# =====================================================================

print("\n" + "=" * 80)
print("MANUSCRIPT-RELEVANT SUMMARY")
print("=" * 80)


print(
    f"\n{EXPERIMENT_LABEL}: "
    "negative images"
)


print(
    f"\nNH = {mean_NH:.3f}"
)

print(
    f"NL = {mean_NL:.3f}"
)

print(
    f"NH - NL = {mean_diff:.3f}, "
    f"95% CI "
    f"[{ci95_low:.3f}, "
    f"{ci95_high:.3f}]"
)

print(
    f"t({df}) = {t_stat:.3f}, "
    f"p = {p_two_sided:.6f}, "
    f"dz = {cohens_dz:.3f}"
)


print(
    f"\nJZS BF01 = "
    f"{BF01_primary:.3f}"
)

print(
    f"JZS BF10 = "
    f"{BF10_primary:.3f}"
)


print(
    "\nTOST sensitivity:"
)

print(
    tost_df[
        [
            "SESOI_dz",
            "raw_lower_bound",
            "raw_upper_bound",
            "TOST_p",
            "CI_low",
            "CI_high",
            "equivalent",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    f"\n{participant_path}"
)

print(
    summary_path
)

print(
    tost_path
)

print(
    bf_path
)


print(
    "\nAnalysis complete."
)

Using participant-level `cell_wide` already present in notebook memory.

NEGATIVE-IMAGE NULL / EQUIVALENCE ANALYSIS

Experiment: Experiment 1
Participants included: 22
Participant IDs:
['pid001', 'pid002', 'pid003', 'pid004', 'pid005', 'pid006', 'pid007', 'pid008', 'pid009', 'pid010', 'pid011', 'pid012', 'pid013', 'pid014', 'pid015', 'pid016', 'pid019', 'pid020', 'pid021', 'pid022', 'pid023', 'pid024']

--------------------------------------------------------------------------------
STANDARD PAIRED COMPARISON
--------------------------------------------------------------------------------

NH mean = -41.597
NL mean = -40.986
NH - NL mean difference = -0.612
SD of paired differences = 4.529
SE of paired difference = 0.966
t(21) = -0.633
p = 0.533331
Cohen's dz = -0.135
95% CI for NH - NL = [-2.620, 1.396]

TOST EQUIVALENCE TESTS

SESOI: dz = ±0.20
Raw equivalence bounds = [-0.906, 0.906]
Lower test: t(21) = 0.305, p = 0.381792
Upper test: t(21) = -1.571, p = 0.065512
TOST p = 0.381792
9

In [17]:
# ================================================================
# SUBJECTIVE-VALENCE-MATCHED SENSITIVITY ANALYSIS
#
# Run this cell immediately after the FIRST/main analysis cell in:
#     Resolution Study v1 (2)(1).ipynb
#
# This analysis does NOT use normative IAPS valence ratings to match
# the stimulus categories.
#
# It instead:
#   1. Calculates each image's observed mean signed rating using
#      participant ratings from BOTH high- and low-resolution
#      presentations.
#   2. Retains the notebook's existing positive/negative stimulus
#      labels to define the intended stimulus categories.
#   3. Removes:
#        - the weakest positive images
#        - the most extremely negative images
#   4. Finds the largest reduced stimulus set in which:
#
#        abs(mean positive observed rating)
#            approximately equals
#        abs(mean negative observed rating)
#
#   5. Reruns the 2 × 2 repeated-measures ANOVA and follow-up tests
#      for:
#        - the complete/full stimulus set
#        - the subjective-valence-matched stimulus subset
#   6. Compares the full and matched statistical patterns.
#
# IMPORTANT INTERPRETIVE NOTE
# -----------------------------
# This is a reviewer-requested sensitivity analysis. Because the
# observed outcome ratings are used to select the stimulus subset,
# it should not be described as a fully independent confirmatory
# analysis.
# ================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.anova import AnovaRM


# ================================================================
# USER-ADJUSTABLE SETTINGS
# ================================================================

# On a roughly -100 to +100 subjective rating scale, this requires
# the absolute positive and negative category means to differ by no
# more than one rating point.
MATCH_TOLERANCE = 1.0

# At least this proportion of each original category must remain.
MIN_RETAINED_PROPORTION = 0.50

# Minimum number of retained images in each category.
MIN_STIMULI_PER_VALENCE = 4

# Your professor proposed removing images from both categories.
REQUIRE_REMOVAL_FROM_BOTH = True

# When True, the retained positive and negative categories must have
# the same number of unique images.
#
# False is generally preferable because the reviewer's concern is
# unequal mean valence magnitude rather than unequal category size.
REQUIRE_EQUAL_STIMULUS_COUNTS = False

# Select the largest total number of retained images among candidate
# subsets that satisfy MATCH_TOLERANCE.
MATCH_SELECTION = "largest_within_tolerance"

# Bonferroni threshold for the two simple high-vs-low comparisons.
FOLLOWUP_ALPHA = 0.025

# Confidence interval level.
CONFIDENCE_LEVEL = 0.95

# Save outputs in a dedicated folder.
OUTPUT_SUBFOLDER = "subjective_valence_matched_analysis"


# ================================================================
# VERIFY THAT THE MAIN NOTEBOOK PIPELINE HAS RUN
# ================================================================

if "paired" not in globals():
    raise RuntimeError(
        "The main notebook analysis must be run before this cell. "
        "The dataframe `paired` does not currently exist."
    )

if not isinstance(paired, pd.DataFrame):
    raise TypeError(
        "`paired` exists but is not a pandas DataFrame."
    )

if paired.empty:
    raise RuntimeError(
        "`paired` is empty. Check the main data-loading pipeline."
    )

required_columns = {
    "pid",
    "IAPS",
    "valence",
    "enjoy_high",
    "enjoy_low",
}

missing_columns = required_columns.difference(paired.columns)

if missing_columns:
    raise KeyError(
        "`paired` is missing required columns:\n"
        + ", ".join(sorted(missing_columns))
        + "\n\nAvailable columns:\n"
        + ", ".join(map(str, paired.columns))
    )

if "OUTPUT_DIR" not in globals() or OUTPUT_DIR is None:
    if "DATA_DIR" in globals() and DATA_DIR:
        OUTPUT_DIR = os.path.join(
            DATA_DIR,
            "resolution_analysis_outputs",
        )
    else:
        OUTPUT_DIR = os.path.join(
            os.getcwd(),
            "resolution_analysis_outputs",
        )

matched_output_dir = os.path.join(
    OUTPUT_DIR,
    OUTPUT_SUBFOLDER,
)

os.makedirs(
    matched_output_dir,
    exist_ok=True,
)


# ================================================================
# CREATE A CLEAN WORKING DATAFRAME
# ================================================================

subjective_source = paired.copy()

subjective_source["pid"] = (
    subjective_source["pid"]
    .astype(str)
    .str.strip()
)

subjective_source["IAPS"] = pd.to_numeric(
    subjective_source["IAPS"],
    errors="coerce",
).round()

subjective_source["enjoy_high"] = pd.to_numeric(
    subjective_source["enjoy_high"],
    errors="coerce",
)

subjective_source["enjoy_low"] = pd.to_numeric(
    subjective_source["enjoy_low"],
    errors="coerce",
)

subjective_source["valence"] = (
    subjective_source["valence"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Accommodate possible alternative category labels.
subjective_source["valence"] = (
    subjective_source["valence"]
    .replace(
        {
            "positive": "pos",
            "negative": "neg",
            "pleasant": "pos",
            "unpleasant": "neg",
        }
    )
)

subjective_source = subjective_source.dropna(
    subset=[
        "pid",
        "IAPS",
    ]
).copy()

subjective_source["IAPS"] = (
    subjective_source["IAPS"]
    .astype(int)
)

subjective_source = subjective_source[
    subjective_source["valence"].isin(
        ["pos", "neg"]
    )
].copy()

if subjective_source.empty:
    raise RuntimeError(
        "No rows remained after restricting `valence` to "
        "'pos' and 'neg'. Inspect `paired['valence'].unique()`."
    )


# ================================================================
# CREATE A READABLE STIMULUS LABEL
# ================================================================

possible_identifier_columns = [
    "image_id",
    "image_id_x",
    "image_id_y",
    "stimulus_label",
    "filename",
    "file_name",
    "stimulus",
    "stimulus_name",
]

available_identifier_columns = [
    column
    for column in possible_identifier_columns
    if column in subjective_source.columns
]

if available_identifier_columns:

    identifier_column = available_identifier_columns[0]

    subjective_source["stimulus_label"] = (
        subjective_source[identifier_column]
        .astype(str)
        .str.strip()
    )

    invalid_label = (
        subjective_source["stimulus_label"]
        .str.lower()
        .isin(
            [
                "",
                "nan",
                "none",
                "<na>",
            ]
        )
    )

    subjective_source.loc[
        invalid_label,
        "stimulus_label",
    ] = (
        "IAPS_"
        + subjective_source.loc[
            invalid_label,
            "IAPS",
        ].astype(str)
    )

    print(
        f"Using `{identifier_column}` as the readable "
        "stimulus identifier."
    )

else:

    subjective_source["stimulus_label"] = (
        "IAPS_"
        + subjective_source["IAPS"].astype(str)
    )

    print(
        "No separate image-name column was found. "
        "Using IAPS numbers as stimulus identifiers."
    )


# ================================================================
# HELPER FUNCTIONS
# ================================================================

def format_p(p_value):
    """Format p-values for readable console output."""

    if not np.isfinite(p_value):
        return "p = NA"

    if p_value < 0.001:
        return "p < .001"

    return f"p = {p_value:.3f}"


def mean_ci(values, confidence=0.95):
    """Mean, SD, SE, and a t-based confidence interval."""

    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).dropna().to_numpy(dtype=float)

    n = len(values)

    if n == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    mean_value = float(
        np.mean(values)
    )

    if n == 1:
        return {
            "n": 1,
            "mean": mean_value,
            "sd": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    sd_value = float(
        np.std(values, ddof=1)
    )

    se_value = (
        sd_value / np.sqrt(n)
    )

    critical_t = stats.t.ppf(
        1 - ((1 - confidence) / 2),
        df=n - 1,
    )

    return {
        "n": n,
        "mean": mean_value,
        "sd": sd_value,
        "se": se_value,
        "ci_low": (
            mean_value
            - critical_t * se_value
        ),
        "ci_high": (
            mean_value
            + critical_t * se_value
        ),
    }


def paired_test(
    high_values,
    low_values,
    confidence=0.95,
):
    """
    Paired high-versus-low t-test with Cohen's dz and a CI
    for the high-minus-low difference.
    """

    pair = pd.DataFrame(
        {
            "high": pd.to_numeric(
                pd.Series(high_values),
                errors="coerce",
            ),
            "low": pd.to_numeric(
                pd.Series(low_values),
                errors="coerce",
            ),
        }
    ).dropna()

    high = pair["high"].to_numpy(dtype=float)
    low = pair["low"].to_numpy(dtype=float)

    difference = high - low
    n = len(difference)

    if n < 2:
        return {
            "n": n,
            "mean_high": np.nan,
            "mean_low": np.nan,
            "mean_difference": np.nan,
            "sd_difference": np.nan,
            "se_difference": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
            "dz": np.nan,
        }

    difference_summary = mean_ci(
        difference,
        confidence=confidence,
    )

    t_value, p_value = stats.ttest_rel(
        high,
        low,
        nan_policy="omit",
    )

    sd_difference = difference_summary["sd"]

    dz_value = (
        difference_summary["mean"]
        / sd_difference
        if (
            np.isfinite(sd_difference)
            and sd_difference > 0
        )
        else np.nan
    )

    return {
        "n": n,
        "mean_high": float(np.mean(high)),
        "mean_low": float(np.mean(low)),
        "mean_difference": (
            difference_summary["mean"]
        ),
        "sd_difference": (
            difference_summary["sd"]
        ),
        "se_difference": (
            difference_summary["se"]
        ),
        "ci_low": (
            difference_summary["ci_low"]
        ),
        "ci_high": (
            difference_summary["ci_high"]
        ),
        "t": float(t_value),
        "df": n - 1,
        "p": float(p_value),
        "dz": (
            float(dz_value)
            if np.isfinite(dz_value)
            else np.nan
        ),
    }


def one_sample_test(
    values,
    null_value=0.0,
    confidence=0.95,
):
    """One-sample t-test with Cohen's dz and a t-based CI."""

    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce",
    ).dropna().to_numpy(dtype=float)

    n = len(values)

    if n < 2:
        return {
            "n": n,
            "mean": np.nan,
            "sd": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
            "dz": np.nan,
        }

    summary = mean_ci(
        values,
        confidence=confidence,
    )

    t_value, p_value = stats.ttest_1samp(
        values,
        popmean=null_value,
        nan_policy="omit",
    )

    dz_value = (
        (
            summary["mean"]
            - null_value
        )
        / summary["sd"]
        if (
            np.isfinite(summary["sd"])
            and summary["sd"] > 0
        )
        else np.nan
    )

    return {
        **summary,
        "t": float(t_value),
        "df": n - 1,
        "p": float(p_value),
        "dz": (
            float(dz_value)
            if np.isfinite(dz_value)
            else np.nan
        ),
    }


def extract_anova_table(
    fitted_anova,
    dataset_label,
):
    """Convert a statsmodels AnovaRM table to a clean dataframe."""

    if fitted_anova is None:
        return pd.DataFrame()

    table = (
        fitted_anova.anova_table
        .reset_index()
        .rename(
            columns={
                "index": "effect",
                "F Value": "F",
                "Num DF": "df_num",
                "Den DF": "df_den",
                "Pr > F": "p",
            }
        )
    )

    table.insert(
        0,
        "dataset",
        dataset_label,
    )

    return table


def analyze_dataset(
    source_df,
    dataset_label,
):
    """
    Calculate participant cell means, run a 2×2 repeated-measures
    ANOVA, and run simple effects and the direct interaction contrast.
    """

    source_df = source_df.copy()

    long_df = pd.concat(
        [
            source_df[
                [
                    "pid",
                    "IAPS",
                    "valence",
                    "enjoy_high",
                ]
            ]
            .rename(
                columns={
                    "enjoy_high": "rating"
                }
            )
            .assign(
                resolution="high"
            ),

            source_df[
                [
                    "pid",
                    "IAPS",
                    "valence",
                    "enjoy_low",
                ]
            ]
            .rename(
                columns={
                    "enjoy_low": "rating"
                }
            )
            .assign(
                resolution="low"
            ),
        ],
        ignore_index=True,
    )

    long_df["rating"] = pd.to_numeric(
        long_df["rating"],
        errors="coerce",
    )

    long_df = long_df.dropna(
        subset=[
            "pid",
            "IAPS",
            "valence",
            "resolution",
            "rating",
        ]
    ).copy()

    cell_means = (
        long_df
        .groupby(
            [
                "pid",
                "valence",
                "resolution",
            ],
            as_index=False,
        )
        .agg(
            mean_rating=(
                "rating",
                "mean",
            ),
            n_observations=(
                "rating",
                "size",
            ),
            n_unique_stimuli=(
                "IAPS",
                "nunique",
            ),
        )
    )

    cell_wide = (
        cell_means
        .pivot(
            index="pid",
            columns=[
                "valence",
                "resolution",
            ],
            values="mean_rating",
        )
    )

    required_cells = [
        ("pos", "high"),
        ("pos", "low"),
        ("neg", "high"),
        ("neg", "low"),
    ]

    for cell in required_cells:
        if cell not in cell_wide.columns:
            cell_wide[cell] = np.nan

    cell_wide = (
        cell_wide[
            required_cells
        ]
        .copy()
    )

    cell_wide.columns = [
        "PH",
        "PL",
        "NH",
        "NL",
    ]

    cell_wide = (
        cell_wide
        .reset_index()
        .dropna(
            subset=[
                "PH",
                "PL",
                "NH",
                "NL",
            ]
        )
        .copy()
    )

    complete_pids = set(
        cell_wide["pid"]
    )

    cell_means_complete = (
        cell_means[
            cell_means["pid"].isin(
                complete_pids
            )
        ]
        .copy()
    )

    if len(cell_wide) < 2:
        raise RuntimeError(
            f"{dataset_label}: fewer than two participants "
            "had complete PH, PL, NH, and NL cells."
        )

    anova_input = (
        cell_means_complete[
            [
                "pid",
                "valence",
                "resolution",
                "mean_rating",
            ]
        ]
        .copy()
    )

    try:
        anova_result = AnovaRM(
            data=anova_input,
            depvar="mean_rating",
            subject="pid",
            within=[
                "valence",
                "resolution",
            ],
        ).fit()

    except Exception as exc:
        print(
            f"\nANOVA warning for {dataset_label}: "
            f"{exc}"
        )
        anova_result = None

    positive_test = paired_test(
        cell_wide["PH"],
        cell_wide["PL"],
        confidence=CONFIDENCE_LEVEL,
    )

    negative_test = paired_test(
        cell_wide["NH"],
        cell_wide["NL"],
        confidence=CONFIDENCE_LEVEL,
    )

    cell_wide[
        "positive_resolution_effect"
    ] = (
        cell_wide["PH"]
        - cell_wide["PL"]
    )

    cell_wide[
        "negative_resolution_effect"
    ] = (
        cell_wide["NH"]
        - cell_wide["NL"]
    )

    cell_wide[
        "interaction_contrast"
    ] = (
        cell_wide[
            "positive_resolution_effect"
        ]
        - cell_wide[
            "negative_resolution_effect"
        ]
    )

    interaction_test = one_sample_test(
        cell_wide["interaction_contrast"],
        null_value=0.0,
        confidence=CONFIDENCE_LEVEL,
    )

    descriptive_rows = []

    for (
        cell_name,
        valence_name,
        resolution_name,
    ) in [
        ("PH", "positive", "high"),
        ("PL", "positive", "low"),
        ("NH", "negative", "high"),
        ("NL", "negative", "low"),
    ]:

        summary = mean_ci(
            cell_wide[cell_name],
            confidence=CONFIDENCE_LEVEL,
        )

        descriptive_rows.append(
            {
                "dataset": dataset_label,
                "cell": cell_name,
                "valence": valence_name,
                "resolution": resolution_name,
                **summary,
            }
        )

    descriptives = pd.DataFrame(
        descriptive_rows
    )

    inferential_rows = pd.DataFrame(
        [
            {
                "dataset": dataset_label,
                "analysis": (
                    "Positive high versus low"
                ),
                **positive_test,
            },
            {
                "dataset": dataset_label,
                "analysis": (
                    "Negative high versus low"
                ),
                **negative_test,
            },
            {
                "dataset": dataset_label,
                "analysis": (
                    "Interaction contrast: "
                    "(PH-PL)-(NH-NL)"
                ),
                **interaction_test,
            },
        ]
    )

    return {
        "label": dataset_label,
        "long": long_df,
        "cell_means": (
            cell_means_complete
        ),
        "cell_wide": cell_wide,
        "anova": anova_result,
        "anova_table": extract_anova_table(
            anova_result,
            dataset_label,
        ),
        "positive_test": positive_test,
        "negative_test": negative_test,
        "interaction_test": interaction_test,
        "descriptives": descriptives,
        "inferential": inferential_rows,
    }


def print_analysis_results(result):
    """Print ANOVA and planned comparisons."""

    label = result["label"]

    print("\n" + "=" * 78)
    print(label.upper())
    print("=" * 78)

    print(
        "Participants with complete PH, PL, NH, "
        f"and NL cells: {len(result['cell_wide'])}"
    )

    print("\nRepeated-measures ANOVA:")

    if result["anova"] is not None:
        print(result["anova"])
    else:
        print("ANOVA could not be estimated.")

    positive = result["positive_test"]
    negative = result["negative_test"]
    interaction = result["interaction_test"]

    print(
        "\nPositive stimuli: high versus low resolution"
    )
    print(
        f"M_high = {positive['mean_high']:.3f}, "
        f"M_low = {positive['mean_low']:.3f}"
    )
    print(
        f"High − low = {positive['mean_difference']:.3f}, "
        f"{int(CONFIDENCE_LEVEL * 100)}% CI "
        f"[{positive['ci_low']:.3f}, "
        f"{positive['ci_high']:.3f}], "
        f"t({positive['df']}) = "
        f"{positive['t']:.3f}, "
        f"{format_p(positive['p'])}, "
        f"dz = {positive['dz']:.3f}"
    )

    print(
        "\nNegative stimuli: high versus low resolution"
    )
    print(
        f"M_high = {negative['mean_high']:.3f}, "
        f"M_low = {negative['mean_low']:.3f}"
    )
    print(
        f"High − low = {negative['mean_difference']:.3f}, "
        f"{int(CONFIDENCE_LEVEL * 100)}% CI "
        f"[{negative['ci_low']:.3f}, "
        f"{negative['ci_high']:.3f}], "
        f"t({negative['df']}) = "
        f"{negative['t']:.3f}, "
        f"{format_p(negative['p'])}, "
        f"dz = {negative['dz']:.3f}"
    )

    print(
        "\nDirect interaction contrast: "
        "(PH − PL) − (NH − NL)"
    )
    print(
        f"M = {interaction['mean']:.3f}, "
        f"{int(CONFIDENCE_LEVEL * 100)}% CI "
        f"[{interaction['ci_low']:.3f}, "
        f"{interaction['ci_high']:.3f}], "
        f"t({interaction['df']}) = "
        f"{interaction['t']:.3f}, "
        f"{format_p(interaction['p'])}, "
        f"dz = {interaction['dz']:.3f}"
    )


# ================================================================
# COMPUTE OBSERVED SUBJECTIVE VALENCE FOR EACH IMAGE
# ================================================================
#
# For each participant × image row, average the available high- and
# low-resolution ratings. This ensures that an image's matching score
# is not based on only one resolution condition.
#
# Then average those participant-level image means across participants.

subjective_source[
    "participant_image_mean_rating"
] = (
    subjective_source[
        [
            "enjoy_high",
            "enjoy_low",
        ]
    ]
    .mean(
        axis=1,
        skipna=True,
    )
)

subjective_source.loc[
    subjective_source[
        [
            "enjoy_high",
            "enjoy_low",
        ]
    ].notna().sum(axis=1) == 0,
    "participant_image_mean_rating",
] = np.nan

stimulus_subjective_summary = (
    subjective_source
    .dropna(
        subset=[
            "participant_image_mean_rating",
        ]
    )
    .groupby(
        [
            "IAPS",
            "valence",
        ],
        as_index=False,
    )
    .agg(
        stimulus_label=(
            "stimulus_label",
            lambda values: (
                values.dropna().iloc[0]
                if len(values.dropna()) > 0
                else ""
            ),
        ),
        observed_mean_rating=(
            "participant_image_mean_rating",
            "mean",
        ),
        observed_sd_rating=(
            "participant_image_mean_rating",
            "std",
        ),
        observed_se_rating=(
            "participant_image_mean_rating",
            lambda values: (
                values.std(ddof=1)
                / np.sqrt(values.notna().sum())
                if values.notna().sum() > 1
                else np.nan
            ),
        ),
        n_participants_rating_image=(
            "pid",
            "nunique",
        ),
        observed_mean_high=(
            "enjoy_high",
            "mean",
        ),
        observed_mean_low=(
            "enjoy_low",
            "mean",
        ),
    )
)

# Preserve the intended stimulus categories, but identify category
# inconsistencies in the observed means.
stimulus_subjective_summary[
    "observed_sign_consistent"
] = np.select(
    [
        (
            stimulus_subjective_summary["valence"]
            == "pos"
        )
        & (
            stimulus_subjective_summary[
                "observed_mean_rating"
            ] > 0
        ),

        (
            stimulus_subjective_summary["valence"]
            == "neg"
        )
        & (
            stimulus_subjective_summary[
                "observed_mean_rating"
            ] < 0
        ),
    ],
    [
        True,
        True,
    ],
    default=False,
)

sign_inconsistent = (
    stimulus_subjective_summary[
        ~stimulus_subjective_summary[
            "observed_sign_consistent"
        ]
    ]
    .copy()
)

print("\n" + "=" * 78)
print("OBSERVED STIMULUS-LEVEL SUBJECTIVE RATINGS")
print("=" * 78)

print(
    "Unique images with observed ratings: "
    f"{stimulus_subjective_summary['IAPS'].nunique()}"
)

print(
    "Images whose observed rating sign did not match "
    "their assigned positive/negative category: "
    f"{len(sign_inconsistent)}"
)

if not sign_inconsistent.empty:
    print(
        "\nThese sign-inconsistent images remain categorized "
        "according to the experiment's original category labels:"
    )
    print(
        sign_inconsistent[
            [
                "IAPS",
                "stimulus_label",
                "valence",
                "observed_mean_rating",
                "n_participants_rating_image",
            ]
        ]
        .sort_values(
            [
                "valence",
                "observed_mean_rating",
            ]
        )
        .to_string(index=False)
    )


# ================================================================
# CREATE POSITIVE AND NEGATIVE STIMULUS TABLES
# ================================================================

positive_stimuli = (
    stimulus_subjective_summary[
        stimulus_subjective_summary[
            "valence"
        ] == "pos"
    ]
    .sort_values(
        [
            "observed_mean_rating",
            "IAPS",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

negative_stimuli = (
    stimulus_subjective_summary[
        stimulus_subjective_summary[
            "valence"
        ] == "neg"
    ]
    .sort_values(
        [
            "observed_mean_rating",
            "IAPS",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

n_positive_original = len(
    positive_stimuli
)

n_negative_original = len(
    negative_stimuli
)

if (
    n_positive_original
    < MIN_STIMULI_PER_VALENCE
):
    raise RuntimeError(
        f"Only {n_positive_original} positive images "
        "had usable subjective ratings."
    )

if (
    n_negative_original
    < MIN_STIMULI_PER_VALENCE
):
    raise RuntimeError(
        f"Only {n_negative_original} negative images "
        "had usable subjective ratings."
    )

original_positive_mean = (
    positive_stimuli[
        "observed_mean_rating"
    ].mean()
)

original_negative_mean = (
    negative_stimuli[
        "observed_mean_rating"
    ].mean()
)

original_magnitude_difference = abs(
    abs(original_positive_mean)
    - abs(original_negative_mean)
)

print("\nOriginal observed subjective means:")

print(
    f"Positive images: n = {n_positive_original}, "
    f"M = {original_positive_mean:.3f}, "
    f"|M| = {abs(original_positive_mean):.3f}"
)

print(
    f"Negative images: n = {n_negative_original}, "
    f"M = {original_negative_mean:.3f}, "
    f"|M| = {abs(original_negative_mean):.3f}"
)

print(
    "Original absolute-magnitude difference = "
    f"{original_magnitude_difference:.3f}"
)


# ================================================================
# DEFINE MINIMUM RETAINED COUNTS
# ================================================================

minimum_positive_retained = max(
    MIN_STIMULI_PER_VALENCE,
    int(
        np.ceil(
            n_positive_original
            * MIN_RETAINED_PROPORTION
        )
    ),
)

minimum_negative_retained = max(
    MIN_STIMULI_PER_VALENCE,
    int(
        np.ceil(
            n_negative_original
            * MIN_RETAINED_PROPORTION
        )
    ),
)

maximum_positive_retained = (
    n_positive_original - 1
    if REQUIRE_REMOVAL_FROM_BOTH
    else n_positive_original
)

maximum_negative_retained = (
    n_negative_original - 1
    if REQUIRE_REMOVAL_FROM_BOTH
    else n_negative_original
)

if (
    maximum_positive_retained
    < minimum_positive_retained
):
    raise RuntimeError(
        "Positive category is too small to satisfy both "
        "the minimum-retention rule and required removal."
    )

if (
    maximum_negative_retained
    < minimum_negative_retained
):
    raise RuntimeError(
        "Negative category is too small to satisfy both "
        "the minimum-retention rule and required removal."
    )


# ================================================================
# SEARCH ALL DIRECTIONAL TRIMMING SOLUTIONS
#
# Positive:
#   sorted weakest to strongest positive
#   retain the tail -> removes weakest positive images
#
# Negative:
#   sorted most negative to least negative
#   retain the tail -> removes most extremely negative images
# ================================================================

candidate_rows = []

for n_positive_keep in range(
    minimum_positive_retained,
    maximum_positive_retained + 1,
):

    positive_keep = (
        positive_stimuli
        .tail(n_positive_keep)
        .copy()
    )

    positive_mean = (
        positive_keep[
            "observed_mean_rating"
        ].mean()
    )

    for n_negative_keep in range(
        minimum_negative_retained,
        maximum_negative_retained + 1,
    ):

        if (
            REQUIRE_EQUAL_STIMULUS_COUNTS
            and n_positive_keep
            != n_negative_keep
        ):
            continue

        negative_keep = (
            negative_stimuli
            .tail(n_negative_keep)
            .copy()
        )

        negative_mean = (
            negative_keep[
                "observed_mean_rating"
            ].mean()
        )

        positive_magnitude = abs(
            positive_mean
        )

        negative_magnitude = abs(
            negative_mean
        )

        magnitude_difference = abs(
            positive_magnitude
            - negative_magnitude
        )

        candidate_rows.append(
            {
                "n_positive_original":
                    n_positive_original,

                "n_negative_original":
                    n_negative_original,

                "n_positive_retained":
                    n_positive_keep,

                "n_negative_retained":
                    n_negative_keep,

                "n_total_retained":
                    (
                        n_positive_keep
                        + n_negative_keep
                    ),

                "n_positive_removed":
                    (
                        n_positive_original
                        - n_positive_keep
                    ),

                "n_negative_removed":
                    (
                        n_negative_original
                        - n_negative_keep
                    ),

                "positive_signed_mean":
                    positive_mean,

                "negative_signed_mean":
                    negative_mean,

                "positive_absolute_mean":
                    positive_magnitude,

                "negative_absolute_mean":
                    negative_magnitude,

                "absolute_magnitude_difference":
                    magnitude_difference,

                "within_requested_tolerance":
                    (
                        magnitude_difference
                        <= MATCH_TOLERANCE
                    ),

                "weakest_retained_positive":
                    positive_keep[
                        "observed_mean_rating"
                    ].min(),

                "most_negative_retained":
                    negative_keep[
                        "observed_mean_rating"
                    ].min(),
            }
        )

candidate_solutions = pd.DataFrame(
    candidate_rows
)

if candidate_solutions.empty:
    raise RuntimeError(
        "No candidate matching solutions were generated. "
        "Reduce MIN_RETAINED_PROPORTION, disable equal-count "
        "matching, or allow retention of all images."
    )


# ================================================================
# SELECT THE MATCHED SOLUTION
# ================================================================

within_tolerance = (
    candidate_solutions[
        candidate_solutions[
            "within_requested_tolerance"
        ]
    ]
    .copy()
)

if (
    MATCH_SELECTION
    == "largest_within_tolerance"
):

    if not within_tolerance.empty:

        selected_solution = (
            within_tolerance
            .sort_values(
                [
                    "n_total_retained",
                    "absolute_magnitude_difference",
                    "n_positive_removed",
                    "n_negative_removed",
                ],
                ascending=[
                    False,
                    True,
                    True,
                    True,
                ],
            )
            .iloc[0]
        )

        selection_message = (
            "Selected the largest retained stimulus set "
            f"with an absolute mean-magnitude difference "
            f"≤ {MATCH_TOLERANCE:.3f}."
        )

    else:

        selected_solution = (
            candidate_solutions
            .sort_values(
                [
                    "absolute_magnitude_difference",
                    "n_total_retained",
                    "n_positive_removed",
                    "n_negative_removed",
                ],
                ascending=[
                    True,
                    False,
                    True,
                    True,
                ],
            )
            .iloc[0]
        )

        selection_message = (
            "No candidate reached the requested tolerance. "
            "Selected the closest available subjective-valence "
            "match, with retained sample size used as a tiebreaker."
        )

elif MATCH_SELECTION == "closest":

    selected_solution = (
        candidate_solutions
        .sort_values(
            [
                "absolute_magnitude_difference",
                "n_total_retained",
                "n_positive_removed",
                "n_negative_removed",
            ],
            ascending=[
                True,
                False,
                True,
                True,
            ],
        )
        .iloc[0]
    )

    selection_message = (
        "Selected the candidate with the smallest absolute "
        "difference between positive and negative mean magnitudes."
    )

else:

    raise ValueError(
        "MATCH_SELECTION must be either "
        "'largest_within_tolerance' or 'closest'."
    )

n_positive_keep = int(
    selected_solution[
        "n_positive_retained"
    ]
)

n_negative_keep = int(
    selected_solution[
        "n_negative_retained"
    ]
)

# Use the exact same directional rules used in the search.
matched_positive_stimuli = (
    positive_stimuli
    .tail(n_positive_keep)
    .copy()
)

matched_negative_stimuli = (
    negative_stimuli
    .tail(n_negative_keep)
    .copy()
)

matched_stimuli = pd.concat(
    [
        matched_positive_stimuli,
        matched_negative_stimuli,
    ],
    ignore_index=True,
)

matched_iaps_ids = set(
    matched_stimuli[
        "IAPS"
    ].astype(int)
)

excluded_stimuli = (
    stimulus_subjective_summary[
        ~stimulus_subjective_summary[
            "IAPS"
        ].isin(matched_iaps_ids)
    ]
    .copy()
    .sort_values(
        [
            "valence",
            "observed_mean_rating",
            "IAPS",
        ]
    )
)

matched_positive_mean = (
    matched_positive_stimuli[
        "observed_mean_rating"
    ].mean()
)

matched_negative_mean = (
    matched_negative_stimuli[
        "observed_mean_rating"
    ].mean()
)

matched_magnitude_difference = abs(
    abs(matched_positive_mean)
    - abs(matched_negative_mean)
)


# ================================================================
# REPORT THE MATCH
# ================================================================

print("\n" + "=" * 78)
print("SELECTED SUBJECTIVE-VALENCE MATCH")
print("=" * 78)

print(selection_message)

print(
    f"\nPositive images retained: "
    f"{len(matched_positive_stimuli)} / "
    f"{n_positive_original}"
)

print(
    f"Negative images retained: "
    f"{len(matched_negative_stimuli)} / "
    f"{n_negative_original}"
)

print(
    f"Positive images removed: "
    f"{n_positive_original - len(matched_positive_stimuli)}"
)

print(
    f"Negative images removed: "
    f"{n_negative_original - len(matched_negative_stimuli)}"
)

print(
    "\nOriginal signed means:"
)

print(
    f"  Positive M = {original_positive_mean:.3f}; "
    f"|M| = {abs(original_positive_mean):.3f}"
)

print(
    f"  Negative M = {original_negative_mean:.3f}; "
    f"|M| = {abs(original_negative_mean):.3f}"
)

print(
    f"  Magnitude difference = "
    f"{original_magnitude_difference:.3f}"
)

print(
    "\nMatched signed means:"
)

print(
    f"  Positive M = {matched_positive_mean:.3f}; "
    f"|M| = {abs(matched_positive_mean):.3f}"
)

print(
    f"  Negative M = {matched_negative_mean:.3f}; "
    f"|M| = {abs(matched_negative_mean):.3f}"
)

print(
    f"  Magnitude difference = "
    f"{matched_magnitude_difference:.3f}"
)

print("\nRetained positive images:")

print(
    matched_positive_stimuli[
        [
            "IAPS",
            "stimulus_label",
            "observed_mean_rating",
            "observed_sd_rating",
            "n_participants_rating_image",
        ]
    ]
    .sort_values(
        "observed_mean_rating"
    )
    .to_string(index=False)
)

print("\nRetained negative images:")

print(
    matched_negative_stimuli[
        [
            "IAPS",
            "stimulus_label",
            "observed_mean_rating",
            "observed_sd_rating",
            "n_participants_rating_image",
        ]
    ]
    .sort_values(
        "observed_mean_rating"
    )
    .to_string(index=False)
)

print("\nExcluded images:")

if excluded_stimuli.empty:

    print(
        "No images were excluded."
    )

else:

    print(
        excluded_stimuli[
            [
                "IAPS",
                "stimulus_label",
                "valence",
                "observed_mean_rating",
                "observed_sd_rating",
                "n_participants_rating_image",
            ]
        ]
        .to_string(index=False)
    )


# ================================================================
# CREATE FULL AND MATCHED PARTICIPANT DATASETS
# ================================================================

full_analysis_data = (
    subjective_source[
        subjective_source[
            "valence"
        ].isin(["pos", "neg"])
    ]
    .copy()
)

matched_analysis_data = (
    subjective_source[
        subjective_source[
            "IAPS"
        ].isin(matched_iaps_ids)
    ]
    .copy()
)

print("\n" + "=" * 78)
print("PARTICIPANT DATASETS")
print("=" * 78)

print(
    "Full participant × image rows: "
    f"{len(full_analysis_data)}"
)

print(
    "Matched participant × image rows: "
    f"{len(matched_analysis_data)}"
)

print(
    "Full unique images: "
    f"{full_analysis_data['IAPS'].nunique()}"
)

print(
    "Matched unique images: "
    f"{matched_analysis_data['IAPS'].nunique()}"
)

print(
    "Full unique participants: "
    f"{full_analysis_data['pid'].nunique()}"
)

print(
    "Matched unique participants: "
    f"{matched_analysis_data['pid'].nunique()}"
)


# ================================================================
# RUN FULL AND MATCHED ANALYSES
# ================================================================

full_results = analyze_dataset(
    full_analysis_data,
    dataset_label="Full stimulus set",
)

matched_results = analyze_dataset(
    matched_analysis_data,
    dataset_label=(
        "Subjective-valence-matched subset"
    ),
)

print_analysis_results(
    full_results
)

print_analysis_results(
    matched_results
)


# ================================================================
# COMPARE FULL AND MATCHED PATTERNS
# ================================================================

comparison_rows = []

for result in [
    full_results,
    matched_results,
]:

    label = result["label"]
    positive = result["positive_test"]
    negative = result["negative_test"]
    interaction = result["interaction_test"]

    anova_table = result["anova_table"]

    if not anova_table.empty:

        interaction_rows = anova_table[
            anova_table["effect"].astype(str)
            .str.contains(
                "valence:resolution",
                case=False,
                regex=False,
            )
        ]

        if interaction_rows.empty:
            interaction_rows = anova_table[
                anova_table["effect"].astype(str)
                .str.contains(
                    "resolution:valence",
                    case=False,
                    regex=False,
                )
            ]

        if not interaction_rows.empty:
            anova_interaction_F = float(
                interaction_rows.iloc[0]["F"]
            )
            anova_interaction_p = float(
                interaction_rows.iloc[0]["p"]
            )
        else:
            anova_interaction_F = np.nan
            anova_interaction_p = np.nan

    else:
        anova_interaction_F = np.nan
        anova_interaction_p = np.nan

    comparison_rows.append(
        {
            "dataset": label,
            "n_participants": (
                len(result["cell_wide"])
            ),
            "n_images": (
                full_analysis_data["IAPS"].nunique()
                if label == "Full stimulus set"
                else matched_analysis_data[
                    "IAPS"
                ].nunique()
            ),
            "positive_high_mean": (
                positive["mean_high"]
            ),
            "positive_low_mean": (
                positive["mean_low"]
            ),
            "positive_high_minus_low": (
                positive["mean_difference"]
            ),
            "positive_simple_p": (
                positive["p"]
            ),
            "positive_simple_dz": (
                positive["dz"]
            ),
            "negative_high_mean": (
                negative["mean_high"]
            ),
            "negative_low_mean": (
                negative["mean_low"]
            ),
            "negative_high_minus_low": (
                negative["mean_difference"]
            ),
            "negative_simple_p": (
                negative["p"]
            ),
            "negative_simple_dz": (
                negative["dz"]
            ),
            "interaction_contrast": (
                interaction["mean"]
            ),
            "interaction_contrast_p": (
                interaction["p"]
            ),
            "interaction_contrast_dz": (
                interaction["dz"]
            ),
            "anova_interaction_F": (
                anova_interaction_F
            ),
            "anova_interaction_p": (
                anova_interaction_p
            ),
        }
    )

full_vs_matched_comparison = pd.DataFrame(
    comparison_rows
)

print("\n" + "=" * 78)
print("FULL VERSUS MATCHED COMPARISON")
print("=" * 78)

print(
    full_vs_matched_comparison.to_string(
        index=False
    )
)


# ================================================================
# AUTOMATED DIRECTIONAL TREND CHECK
# ================================================================

full_positive_effect = (
    full_results[
        "positive_test"
    ]["mean_difference"]
)

matched_positive_effect = (
    matched_results[
        "positive_test"
    ]["mean_difference"]
)

full_negative_effect = (
    full_results[
        "negative_test"
    ]["mean_difference"]
)

matched_negative_effect = (
    matched_results[
        "negative_test"
    ]["mean_difference"]
)

full_interaction_effect = (
    full_results[
        "interaction_test"
    ]["mean"]
)

matched_interaction_effect = (
    matched_results[
        "interaction_test"
    ]["mean"]
)

positive_same_direction = (
    np.sign(full_positive_effect)
    == np.sign(matched_positive_effect)
)

negative_same_direction = (
    np.sign(full_negative_effect)
    == np.sign(matched_negative_effect)
)

interaction_same_direction = (
    np.sign(full_interaction_effect)
    == np.sign(matched_interaction_effect)
)

all_effects_same_direction = (
    positive_same_direction
    and negative_same_direction
    and interaction_same_direction
)

trend_check = pd.DataFrame(
    [
        {
            "effect": (
                "Positive high-minus-low"
            ),
            "full_effect": (
                full_positive_effect
            ),
            "matched_effect": (
                matched_positive_effect
            ),
            "same_direction": (
                positive_same_direction
            ),
        },
        {
            "effect": (
                "Negative high-minus-low"
            ),
            "full_effect": (
                full_negative_effect
            ),
            "matched_effect": (
                matched_negative_effect
            ),
            "same_direction": (
                negative_same_direction
            ),
        },
        {
            "effect": (
                "Valence-by-resolution interaction contrast"
            ),
            "full_effect": (
                full_interaction_effect
            ),
            "matched_effect": (
                matched_interaction_effect
            ),
            "same_direction": (
                interaction_same_direction
            ),
        },
    ]
)

print("\nDirectional trend comparison:")

print(
    trend_check.to_string(
        index=False
    )
)

if all_effects_same_direction:

    print(
        "\nOVERALL DIRECTIONAL CONCLUSION: "
        "The positive simple effect, negative simple effect, "
        "and valence × resolution interaction all retained "
        "the same direction in the matched subset."
    )

else:

    print(
        "\nOVERALL DIRECTIONAL CONCLUSION: "
        "At least one effect changed direction after matching. "
        "Inspect the comparison table before concluding that "
        "the original trend replicated."
    )


# ================================================================
# PARTICIPANT-LEVEL FULL-VERSUS-MATCHED EFFECT COMPARISON
# ================================================================

participant_effect_comparison = pd.merge(
    full_results["cell_wide"][
        [
            "pid",
            "positive_resolution_effect",
            "negative_resolution_effect",
            "interaction_contrast",
        ]
    ].rename(
        columns={
            "positive_resolution_effect":
                "full_positive_effect",

            "negative_resolution_effect":
                "full_negative_effect",

            "interaction_contrast":
                "full_interaction",
        }
    ),

    matched_results["cell_wide"][
        [
            "pid",
            "positive_resolution_effect",
            "negative_resolution_effect",
            "interaction_contrast",
        ]
    ].rename(
        columns={
            "positive_resolution_effect":
                "matched_positive_effect",

            "negative_resolution_effect":
                "matched_negative_effect",

            "interaction_contrast":
                "matched_interaction",
        }
    ),

    on="pid",
    how="inner",
)

correlation_rows = []

for (
    full_column,
    matched_column,
    effect_label,
) in [
    (
        "full_positive_effect",
        "matched_positive_effect",
        "Positive resolution effect",
    ),
    (
        "full_negative_effect",
        "matched_negative_effect",
        "Negative resolution effect",
    ),
    (
        "full_interaction",
        "matched_interaction",
        "Interaction contrast",
    ),
]:

    pair = (
        participant_effect_comparison[
            [
                full_column,
                matched_column,
            ]
        ]
        .dropna()
    )

    if len(pair) >= 3:

        r_value, p_value = stats.pearsonr(
            pair[full_column],
            pair[matched_column],
        )

    else:

        r_value = np.nan
        p_value = np.nan

    correlation_rows.append(
        {
            "effect": effect_label,
            "n": len(pair),
            "full_matched_r": r_value,
            "p": p_value,
        }
    )

participant_effect_correlations = pd.DataFrame(
    correlation_rows
)

print(
    "\nParticipant-level correspondence between full "
    "and matched effects:"
)

print(
    participant_effect_correlations.to_string(
        index=False
    )
)


# ================================================================
# SAVE OUTPUT TABLES
# ================================================================

stimulus_subjective_summary.to_csv(
    os.path.join(
        matched_output_dir,
        "all_stimulus_observed_subjective_ratings.csv",
    ),
    index=False,
)

matched_stimuli.to_csv(
    os.path.join(
        matched_output_dir,
        "retained_subjective_valence_matched_stimuli.csv",
    ),
    index=False,
)

excluded_stimuli.to_csv(
    os.path.join(
        matched_output_dir,
        "excluded_subjective_valence_stimuli.csv",
    ),
    index=False,
)

candidate_solutions.sort_values(
    [
        "within_requested_tolerance",
        "n_total_retained",
        "absolute_magnitude_difference",
    ],
    ascending=[
        False,
        False,
        True,
    ],
).to_csv(
    os.path.join(
        matched_output_dir,
        "all_subjective_valence_matching_solutions.csv",
    ),
    index=False,
)

selected_solution_df = pd.DataFrame(
    [
        selected_solution.to_dict()
    ]
)

selected_solution_df[
    "match_tolerance"
] = MATCH_TOLERANCE

selected_solution_df[
    "minimum_retained_proportion"
] = MIN_RETAINED_PROPORTION

selected_solution_df[
    "required_removal_from_both"
] = REQUIRE_REMOVAL_FROM_BOTH

selected_solution_df[
    "equal_stimulus_counts_required"
] = REQUIRE_EQUAL_STIMULUS_COUNTS

selected_solution_df[
    "selection_strategy"
] = MATCH_SELECTION

selected_solution_df.to_csv(
    os.path.join(
        matched_output_dir,
        "selected_subjective_valence_matching_solution.csv",
    ),
    index=False,
)

full_results["long"].to_csv(
    os.path.join(
        matched_output_dir,
        "full_sample_long_data.csv",
    ),
    index=False,
)

matched_results["long"].to_csv(
    os.path.join(
        matched_output_dir,
        "matched_subset_long_data.csv",
    ),
    index=False,
)

full_results["cell_wide"].to_csv(
    os.path.join(
        matched_output_dir,
        "full_sample_subject_cellmeans.csv",
    ),
    index=False,
)

matched_results["cell_wide"].to_csv(
    os.path.join(
        matched_output_dir,
        "matched_subset_subject_cellmeans.csv",
    ),
    index=False,
)

combined_anova_results = pd.concat(
    [
        full_results["anova_table"],
        matched_results["anova_table"],
    ],
    ignore_index=True,
)

combined_anova_results.to_csv(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_anova_results.csv",
    ),
    index=False,
)

combined_inferential_results = pd.concat(
    [
        full_results["inferential"],
        matched_results["inferential"],
    ],
    ignore_index=True,
)

combined_inferential_results.to_csv(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_followup_results.csv",
    ),
    index=False,
)

combined_descriptives = pd.concat(
    [
        full_results["descriptives"],
        matched_results["descriptives"],
    ],
    ignore_index=True,
)

combined_descriptives.to_csv(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_group_descriptives.csv",
    ),
    index=False,
)

full_vs_matched_comparison.to_csv(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_effect_comparison.csv",
    ),
    index=False,
)

trend_check.to_csv(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_directional_trend_check.csv",
    ),
    index=False,
)

participant_effect_comparison.to_csv(
    os.path.join(
        matched_output_dir,
        "participant_level_full_vs_matched_effects.csv",
    ),
    index=False,
)

participant_effect_correlations.to_csv(
    os.path.join(
        matched_output_dir,
        "participant_level_full_vs_matched_correlations.csv",
    ),
    index=False,
)

sign_inconsistent.to_csv(
    os.path.join(
        matched_output_dir,
        "stimuli_with_observed_sign_inconsistency.csv",
    ),
    index=False,
)


# ================================================================
# FIGURE 1: BEFORE-VERSUS-AFTER SUBJECTIVE VALENCE DISTRIBUTIONS
# ================================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.scatter(
    np.zeros(len(positive_stimuli)),
    positive_stimuli[
        "observed_mean_rating"
    ],
    alpha=0.35,
    label="Full positive",
)

ax.scatter(
    np.ones(len(negative_stimuli)),
    negative_stimuli[
        "observed_mean_rating"
    ],
    alpha=0.35,
    label="Full negative",
)

ax.scatter(
    np.full(
        len(matched_positive_stimuli),
        2,
    ),
    matched_positive_stimuli[
        "observed_mean_rating"
    ],
    alpha=0.65,
    label="Matched positive",
)

ax.scatter(
    np.full(
        len(matched_negative_stimuli),
        3,
    ),
    matched_negative_stimuli[
        "observed_mean_rating"
    ],
    alpha=0.65,
    label="Matched negative",
)

ax.scatter(
    [
        0,
        1,
        2,
        3,
    ],
    [
        original_positive_mean,
        original_negative_mean,
        matched_positive_mean,
        matched_negative_mean,
    ],
    marker="D",
    s=90,
    zorder=5,
    label="Category mean",
)

ax.axhline(
    0,
    linewidth=1,
    linestyle="--",
)

ax.set_xticks(
    [
        0,
        1,
        2,
        3,
    ]
)

ax.set_xticklabels(
    [
        "Full\npositive",
        "Full\nnegative",
        "Matched\npositive",
        "Matched\nnegative",
    ]
)

ax.set_ylabel(
    "Observed mean signed participant rating"
)

ax.set_xlabel(
    "Stimulus set and assigned valence category"
)

ax.set_title(
    "Observed stimulus valence before and after matching"
)

ax.legend(
    frameon=False,
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        matched_output_dir,
        "subjective_valence_before_after_matching.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    os.path.join(
        matched_output_dir,
        "subjective_valence_before_after_matching.pdf",
    ),
    bbox_inches="tight",
)

plt.show()


# ================================================================
# FIGURE 2: FULL VERSUS MATCHED CONDITION MEANS
# ================================================================

condition_order = [
    "PH",
    "PL",
    "NH",
    "NL",
]

condition_labels = [
    "Positive\nHigh",
    "Positive\nLow",
    "Negative\nHigh",
    "Negative\nLow",
]

full_means = [
    full_results[
        "cell_wide"
    ][cell].mean()
    for cell in condition_order
]

matched_means = [
    matched_results[
        "cell_wide"
    ][cell].mean()
    for cell in condition_order
]

full_ses = [
    (
        full_results[
            "cell_wide"
        ][cell].std(ddof=1)
        / np.sqrt(
            full_results[
                "cell_wide"
            ][cell].notna().sum()
        )
    )
    for cell in condition_order
]

matched_ses = [
    (
        matched_results[
            "cell_wide"
        ][cell].std(ddof=1)
        / np.sqrt(
            matched_results[
                "cell_wide"
            ][cell].notna().sum()
        )
    )
    for cell in condition_order
]

x_positions = np.arange(
    len(condition_order)
)

bar_width = 0.36

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.bar(
    x_positions - bar_width / 2,
    full_means,
    width=bar_width,
    yerr=full_ses,
    capsize=5,
    label="Full stimulus set",
)

ax.bar(
    x_positions + bar_width / 2,
    matched_means,
    width=bar_width,
    yerr=matched_ses,
    capsize=5,
    label="Matched subset",
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    condition_labels
)

ax.set_ylabel(
    "Mean participant rating"
)

ax.set_xlabel(
    "Valence and resolution condition"
)

ax.set_title(
    "Full-sample and subjective-valence-matched results"
)

ax.legend(
    frameon=False,
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_condition_means.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_condition_means.pdf",
    ),
    bbox_inches="tight",
)

plt.show()


# ================================================================
# FIGURE 3: FULL VERSUS MATCHED RESOLUTION EFFECTS
# ================================================================

effect_labels = [
    "Positive\nhigh − low",
    "Negative\nhigh − low",
    "Interaction\ncontrast",
]

full_effects = [
    full_positive_effect,
    full_negative_effect,
    full_interaction_effect,
]

matched_effects = [
    matched_positive_effect,
    matched_negative_effect,
    matched_interaction_effect,
]

x_positions = np.arange(
    len(effect_labels)
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    x_positions,
    full_effects,
    marker="o",
    linewidth=2,
    label="Full stimulus set",
)

ax.plot(
    x_positions,
    matched_effects,
    marker="o",
    linewidth=2,
    label="Matched subset",
)

ax.axhline(
    0,
    linewidth=1,
    linestyle="--",
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    effect_labels
)

ax.set_ylabel(
    "Mean rating difference"
)

ax.set_xlabel(
    "Effect"
)

ax.set_title(
    "Direction and magnitude of full versus matched effects"
)

ax.legend(
    frameon=False,
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_effects.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    os.path.join(
        matched_output_dir,
        "full_vs_matched_effects.pdf",
    ),
    bbox_inches="tight",
)

plt.show()


# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n" + "=" * 78)
print("SUBJECTIVE-VALENCE-MATCHED ANALYSIS COMPLETE")
print("=" * 78)

print(
    "Outputs were saved to:\n"
    f"{matched_output_dir}"
)

print(
    "\nPrimary files:"
    "\n  1. all_stimulus_observed_subjective_ratings.csv"
    "\n  2. retained_subjective_valence_matched_stimuli.csv"
    "\n  3. excluded_subjective_valence_stimuli.csv"
    "\n  4. selected_subjective_valence_matching_solution.csv"
    "\n  5. full_vs_matched_anova_results.csv"
    "\n  6. full_vs_matched_followup_results.csv"
    "\n  7. full_vs_matched_effect_comparison.csv"
    "\n  8. full_vs_matched_directional_trend_check.csv"
    "\n  9. subjective_valence_before_after_matching.png"
    "\n 10. full_vs_matched_condition_means.png"
    "\n 11. full_vs_matched_effects.png"
)

print(
    "\nInterpretation sequence:"
    "\n  - Verify that the matched positive and negative "
    "absolute mean ratings differ by no more than the "
    "specified tolerance."
    "\n  - Compare the full and matched valence × resolution "
    "interactions."
    "\n  - Compare the directions and magnitudes of the "
    "positive and negative high-minus-low effects."
    "\n  - Use the matched analysis as a sensitivity check "
    "for whether the original effect depends on unequal "
    "subjective valence magnitude."
)

Using `image_id_x` as the readable stimulus identifier.

OBSERVED STIMULUS-LEVEL SUBJECTIVE RATINGS
Unique images with observed ratings: 394
Images whose observed rating sign did not match their assigned positive/negative category: 23

These sign-inconsistent images remain categorized according to the experiment's original category labels:
 IAPS stimulus_label valence  observed_mean_rating  n_participants_rating_image
 1505           1505     neg              2.260417                           16
 8010           8010     neg              8.006410                           13
 2590           2590     neg              9.666667                           17
 2455           2455     neg             12.266667                           10
 7560           7560     neg             14.416667                            4
 2345           2345     neg             46.185185                           18
 5532           5532     pos            -33.504902                           17
 1811           18

In [15]:
# ============================================
# Resolution study (2-session within-subject) analysis pipeline
# - Pairs files by pid### in filename/path
# - Detects session from "session1"/"session2" in filename/path
# - Uses cond1 as Session 1 stimulus path, cond2 as Session 2 stimulus path
# - Classifies resolution from stimulus path:
#     high-res  -> contains "/stimuli/" (or "\stimuli\")
#     low-res   -> contains "/degraded stimuli/" (or "\degraded stimuli\")
# - Pulls ratings from "slider.response" and RT from "slider.rt"
# - Produces per-participant + group summaries, paired t-tests, and plots
# ============================================

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib qt

# ---------- USER: SET THESE ----------
DATA_DIR = r"C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\Resolution Study\data v1"  # <-- put your folder path here (leave blank now; fill in later)
FILE_GLOB = "*.csv"  # or e.g. "*Resolution Study*.csv"
OUTPUT_DIR = None  # None -> makes "./resolution_analysis_outputs" inside DATA_DIR (or CWD if DATA_DIR blank)
# -------------------------------------

# PsychoPy column names (as you described)
RATING_COL = "slider.response"
RT_COL = "slider.rt"
COND1_COL = "cond1"  # session 1 image paths
COND2_COL = "cond2"  # session 2 image paths

# Resolution classification keywords
HIGH_TOKEN = "stimuli"
LOW_TOKEN = "degraded stimuli"  # note the space (handle slashes/backslashes below)

# Optional: drop rows where rating is missing
DROP_MISSING_RATINGS = True

# If the same image appears multiple times within a session file, keep the LAST occurrence
KEEP_DUPLICATE_POLICY = "last"  # "first" or "last"

# -------------------- helpers --------------------
def norm_path(p: str) -> str:
    """Normalize to lower-case forward-slash path for robust substring checks."""
    if pd.isna(p):
        return ""
    p = str(p).strip()
    p = p.replace("\\", "/")
    return p.lower()

def get_pid_from_path(path: str) -> str | None:
    m = re.search(r"(pid\d{3})", path, flags=re.IGNORECASE)
    return m.group(1).lower() if m else None

def get_session_from_path(path: str) -> int | None:
    s = path.lower()
    if "session1" in s or "sess1" in s:
        return 1
    if "session2" in s or "sess2" in s:
        return 2
    return None

def classify_resolution(stim_path: str) -> str:
    p = norm_path(stim_path)

    # low-res should win if both tokens appear (rare but possible)
    if f"/{LOW_TOKEN}/" in p or p.endswith(f"/{LOW_TOKEN}") or LOW_TOKEN in p:
        return "low"
    if f"/{HIGH_TOKEN}/" in p or p.endswith(f"/{HIGH_TOKEN}") or HIGH_TOKEN in p:
        return "high"

    # fallback: if it contains "degraded" anywhere, call it low
    if "degraded" in p:
        return "low"
    return "unknown"

def image_id_from_path(stim_path: str) -> str:
    p = norm_path(stim_path)
    base = os.path.basename(p)  # works because we normalized slashes
    base = re.sub(r"\.[a-z0-9]+$", "", base)  # strip extension
    return base

def safe_read_csv(fp: str) -> pd.DataFrame:
    # PsychoPy exports are usually comma-separated; keep defaults; handle odd encodings if needed
    return pd.read_csv(fp)

def paired_ttest(x: np.ndarray, y: np.ndarray):
    """
    Paired t-test with Cohen's dz.
    Requires scipy; if unavailable, returns t/p as NaN but still returns dz.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 2:
        return dict(n=n, t=np.nan, p=np.nan, dz=np.nan, mean_x=np.nan, mean_y=np.nan, mean_diff=np.nan)

    d = x - y
    mean_x, mean_y = float(np.mean(x)), float(np.mean(y))
    mean_diff = float(np.mean(d))
    sd_diff = float(np.std(d, ddof=1)) if n > 1 else np.nan
    dz = mean_diff / sd_diff if (sd_diff and np.isfinite(sd_diff) and sd_diff != 0) else np.nan

    try:
        from scipy import stats
        t, p = stats.ttest_rel(x, y, nan_policy="omit")
        t, p = float(t), float(p)
    except Exception:
        t, p = np.nan, np.nan

    return dict(n=n, t=t, p=p, dz=dz, mean_x=mean_x, mean_y=mean_y, mean_diff=mean_diff)

def mean_sem(a: np.ndarray):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan, np.nan, 0
    m = float(np.mean(a))
    sem = float(np.std(a, ddof=1) / np.sqrt(len(a))) if len(a) > 1 else np.nan
    return m, sem, len(a)

# -------------------- file discovery + pairing --------------------
search_dir = DATA_DIR if DATA_DIR else os.getcwd()
all_files = sorted(glob.glob(os.path.join(search_dir, FILE_GLOB)))

if len(all_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found. Check DATA_DIR and FILE_GLOB.\nDATA_DIR={DATA_DIR!r}, search_dir={search_dir!r}, FILE_GLOB={FILE_GLOB!r}"
    )

pairs = {}  # pid -> {1: fp1, 2: fp2}
unassigned = []

for fp in all_files:
    pid = get_pid_from_path(fp)
    sess = get_session_from_path(fp)
    if pid is None or sess is None:
        unassigned.append(fp)
        continue
    pairs.setdefault(pid, {})
    # keep the latest if duplicates
    pairs[pid][sess] = fp

pids = sorted(pairs.keys())
print(f"Found {len(all_files)} CSVs total.")
print(f"Identified {len(pids)} participant IDs with at least one session file.")
if unassigned:
    print("\nFiles skipped (couldn't detect pid### and/or session1/session2 in path):")
    for fp in unassigned[:20]:
        print("  -", fp)
    if len(unassigned) > 20:
        print(f"  ... plus {len(unassigned)-20} more")

missing = [(pid, pairs[pid].get(1), pairs[pid].get(2)) for pid in pids if (1 not in pairs[pid] or 2 not in pairs[pid])]
if missing:
    print("\nParticipants missing a session file:")
    for pid, s1, s2 in missing:
        print(f"  {pid}: session1={bool(s1)}, session2={bool(s2)}")

# -------------------- parse each session file into long-form trials --------------------
rows = []
for pid in pids:
    for sess in (1, 2):
        fp = pairs[pid].get(sess)
        if not fp:
            continue
        df = safe_read_csv(fp)

        # choose the stimulus column for this session
        stim_col = COND1_COL if sess == 1 else COND2_COL
        if stim_col not in df.columns:
            raise KeyError(
                f"Expected column {stim_col!r} not found in {fp}\n"
                f"Columns present: {list(df.columns)}"
            )

        if RATING_COL not in df.columns or RT_COL not in df.columns:
            raise KeyError(
                f"Expected columns {RATING_COL!r} and/or {RT_COL!r} not found in {fp}\n"
                f"Columns present: {list(df.columns)}"
            )

        tmp = pd.DataFrame({
            "pid": pid,
            "session": sess,
            "stim_path": df[stim_col],
            "rating": pd.to_numeric(df[RATING_COL], errors="coerce"),
            "rt": pd.to_numeric(df[RT_COL], errors="coerce"),
        })

        # drop empty stimulus paths
        tmp["stim_path_norm"] = tmp["stim_path"].map(norm_path)
        tmp = tmp[tmp["stim_path_norm"].astype(bool)]

        # optionally drop missing ratings
        if DROP_MISSING_RATINGS:
            tmp = tmp[np.isfinite(tmp["rating"].to_numpy())]

        tmp["resolution"] = tmp["stim_path"].map(classify_resolution)
        tmp["image_id"] = tmp["stim_path"].map(image_id_from_path)

        # handle duplicates within session for the same image
        if KEEP_DUPLICATE_POLICY in ("first", "last"):
            tmp = tmp.reset_index(drop=False).rename(columns={"index": "_rowidx"})
            tmp = tmp.sort_values("_rowidx", ascending=True)
            tmp = tmp.drop_duplicates(subset=["pid", "session", "image_id"], keep=KEEP_DUPLICATE_POLICY)
            tmp = tmp.drop(columns=["_rowidx"])

        rows.append(tmp[["pid", "session", "image_id", "stim_path", "resolution", "rating", "rt"]])

trials = pd.concat(rows, ignore_index=True)
print(f"\nParsed trials: {len(trials)} rows")
print(trials.head())

# -------------------- within-participant pairing by image_id across sessions --------------------
# wide per participant: one row per image_id with session1 + session2 values
s1 = trials[trials["session"] == 1].rename(columns={
    "stim_path": "stim_s1", "resolution": "res_s1", "rating": "rating_s1", "rt": "rt_s1"
})[["pid", "image_id", "stim_s1", "res_s1", "rating_s1", "rt_s1"]]

s2 = trials[trials["session"] == 2].rename(columns={
    "stim_path": "stim_s2", "resolution": "res_s2", "rating": "rating_s2", "rt": "rt_s2"
})[["pid", "image_id", "stim_s2", "res_s2", "rating_s2", "rt_s2"]]

paired = pd.merge(s1, s2, on=["pid", "image_id"], how="inner")
print(f"\nPaired images across sessions: {len(paired)} rows")

# compute per-image high vs low rating/rt (within-subject, within-image)
def pick_high_low(row):
    # ratings
    rating_high = np.nan
    rating_low = np.nan
    rt_high = np.nan
    rt_low = np.nan

    if row["res_s1"] == "high":
        rating_high, rt_high = row["rating_s1"], row["rt_s1"]
    if row["res_s2"] == "high":
        rating_high, rt_high = row["rating_s2"], row["rt_s2"]

    if row["res_s1"] == "low":
        rating_low, rt_low = row["rating_s1"], row["rt_s1"]
    if row["res_s2"] == "low":
        rating_low, rt_low = row["rating_s2"], row["rt_s2"]

    return pd.Series({
        "rating_high": rating_high,
        "rating_low": rating_low,
        "rt_high": rt_high,
        "rt_low": rt_low,
        "rating_high_minus_low": (rating_high - rating_low) if np.isfinite(rating_high) and np.isfinite(rating_low) else np.nan,
        "rt_low_minus_high": (rt_low - rt_high) if np.isfinite(rt_low) and np.isfinite(rt_high) else np.nan,
        "rating_s2_minus_s1": (row["rating_s2"] - row["rating_s1"]) if np.isfinite(row["rating_s2"]) and np.isfinite(row["rating_s1"]) else np.nan,
        "rt_s2_minus_s1": (row["rt_s2"] - row["rt_s1"]) if np.isfinite(row["rt_s2"]) and np.isfinite(row["rt_s1"]) else np.nan,
    })

paired = pd.concat([paired, paired.apply(pick_high_low, axis=1)], axis=1)

# -------------------- participant-level summaries (used for group stats) --------------------
subj = (
    paired.groupby("pid", as_index=False)
    .agg(
        n_images=("image_id", "count"),

        mean_rating_high=("rating_high", "mean"),
        mean_rating_low=("rating_low", "mean"),
        mean_rating_high_minus_low=("rating_high_minus_low", "mean"),

        mean_rt_high=("rt_high", "mean"),
        mean_rt_low=("rt_low", "mean"),
        mean_rt_low_minus_high=("rt_low_minus_high", "mean"),

        mean_rating_s1=("rating_s1", "mean"),
        mean_rating_s2=("rating_s2", "mean"),
        mean_rating_s2_minus_s1=("rating_s2_minus_s1", "mean"),

        mean_rt_s1=("rt_s1", "mean"),
        mean_rt_s2=("rt_s2", "mean"),
        mean_rt_s2_minus_s1=("rt_s2_minus_s1", "mean"),
    )
)

print("\nParticipant-level summary (first rows):")
print(subj.head())

# -------------------- group-level stats --------------------
# High vs Low (ratings)
test_rating = paired_ttest(subj["mean_rating_high"].to_numpy(), subj["mean_rating_low"].to_numpy())
test_rt = paired_ttest(subj["mean_rt_low"].to_numpy(), subj["mean_rt_high"].to_numpy())  # low minus high framing

# Session 1 vs Session 2 (ratings/rt)
test_sess_rating = paired_ttest(subj["mean_rating_s2"].to_numpy(), subj["mean_rating_s1"].to_numpy())
test_sess_rt = paired_ttest(subj["mean_rt_s2"].to_numpy(), subj["mean_rt_s1"].to_numpy())

# Descriptives
m_hi, sem_hi, n_hi = mean_sem(subj["mean_rating_high"].to_numpy())
m_lo, sem_lo, n_lo = mean_sem(subj["mean_rating_low"].to_numpy())

m_rthi, sem_rthi, _ = mean_sem(subj["mean_rt_high"].to_numpy())
m_rtlo, sem_rtlo, _ = mean_sem(subj["mean_rt_low"].to_numpy())

m_s1, sem_s1, _ = mean_sem(subj["mean_rating_s1"].to_numpy())
m_s2, sem_s2, _ = mean_sem(subj["mean_rating_s2"].to_numpy())

m_rt1, sem_rt1, _ = mean_sem(subj["mean_rt_s1"].to_numpy())
m_rt2, sem_rt2, _ = mean_sem(subj["mean_rt_s2"].to_numpy())

print("\n================ GROUP RESULTS ================")
print(f"N participants (paired): {len(subj)}")
print("\nRatings (High vs Low):")
print(f"  mean_high = {m_hi:.3f} (SEM {sem_hi:.3f}), mean_low = {m_lo:.3f} (SEM {sem_lo:.3f})")
print(f"  paired t({test_rating['n']-1}) = {test_rating['t']:.3f}, p = {test_rating['p']}, Cohen dz = {test_rating['dz']:.3f}")
print(f"  mean(high-low) = {test_rating['mean_diff']:.3f}")

print("\nRT (Low vs High):")
print(f"  mean_low = {m_rtlo:.3f} (SEM {sem_rtlo:.3f}), mean_high = {m_rthi:.3f} (SEM {sem_rthi:.3f})")
print(f"  paired t({test_rt['n']-1}) = {test_rt['t']:.3f}, p = {test_rt['p']}, Cohen dz = {test_rt['dz']:.3f}")
print(f"  mean(low-high) = {test_rt['mean_diff']:.3f}")

print("\nSession effect (Session2 vs Session1) — Ratings:")
print(f"  mean_s2 = {m_s2:.3f} (SEM {sem_s2:.3f}), mean_s1 = {m_s1:.3f} (SEM {sem_s1:.3f})")
print(f"  paired t({test_sess_rating['n']-1}) = {test_sess_rating['t']:.3f}, p = {test_sess_rating['p']}, Cohen dz = {test_sess_rating['dz']:.3f}")
print(f"  mean(s2-s1) = {test_sess_rating['mean_diff']:.3f}")

print("\nSession effect (Session2 vs Session1) — RT:")
print(f"  mean_s2 = {m_rt2:.3f} (SEM {sem_rt2:.3f}), mean_s1 = {m_rt1:.3f} (SEM {sem_rt1:.3f})")
print(f"  paired t({test_sess_rt['n']-1}) = {test_sess_rt['t']:.3f}, p = {test_sess_rt['p']}, Cohen dz = {test_sess_rt['dz']:.3f}")
print(f"  mean(s2-s1) = {test_sess_rt['mean_diff']:.3f}")
print("===============================================")

# -------------------- save outputs --------------------
if OUTPUT_DIR is None:
    base = search_dir if DATA_DIR else os.getcwd()
    OUTPUT_DIR = os.path.join(base, "resolution_analysis_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

trials_out = os.path.join(OUTPUT_DIR, "trials_long.csv")
paired_out = os.path.join(OUTPUT_DIR, "paired_by_image.csv")
subj_out = os.path.join(OUTPUT_DIR, "subject_summary.csv")

trials.to_csv(trials_out, index=False)
paired.to_csv(paired_out, index=False)
subj.to_csv(subj_out, index=False)

print(f"\nSaved:\n  {trials_out}\n  {paired_out}\n  {subj_out}")

# -------------------- plots --------------------
def paired_lines_plot(values_a, values_b, label_a, label_b, ylabel, title, save_name):
    a = np.asarray(values_a, dtype=float)
    b = np.asarray(values_b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]

    fig, ax = plt.subplots(figsize=(7, 5))
    x0, x1 = 0, 1
    for i in range(len(a)):
        ax.plot([x0, x1], [a[i], b[i]], marker="o")
    ax.set_xticks([x0, x1])
    ax.set_xticklabels([label_a, label_b])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()

def bar_sem_plot(means, sems, labels, ylabel, title, save_name):
    fig, ax = plt.subplots(figsize=(7, 5))
    x = np.arange(len(means))
    ax.bar(x, means, yerr=sems, capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()

# Ratings: High vs Low
paired_lines_plot(
    subj["mean_rating_high"], subj["mean_rating_low"],
    "High-res", "Low-res",
    ylabel="Mean rating (slider.response)",
    title="Participant means: High-res vs Low-res ratings",
    save_name="paired_ratings_high_vs_low.png"
)
bar_sem_plot(
    means=[m_hi, m_lo],
    sems=[sem_hi, sem_lo],
    labels=["High-res", "Low-res"],
    ylabel="Mean rating (slider.response)",
    title="Group mean ratings (High-res vs Low-res)",
    save_name="group_bar_ratings_high_vs_low.png"
)

# RT: Low vs High
paired_lines_plot(
    subj["mean_rt_low"], subj["mean_rt_high"],
    "Low-res", "High-res",
    ylabel="Mean RT (slider.rt, seconds)",
    title="Participant means: Low-res vs High-res RT",
    save_name="paired_rt_low_vs_high.png"
)
bar_sem_plot(
    means=[m_rtlo, m_rthi],
    sems=[sem_rtlo, sem_rthi],
    labels=["Low-res", "High-res"],
    ylabel="Mean RT (slider.rt, seconds)",
    title="Group mean RT (Low-res vs High-res)",
    save_name="group_bar_rt_low_vs_high.png"
)

# Session effects
paired_lines_plot(
    subj["mean_rating_s1"], subj["mean_rating_s2"],
    "Session 1", "Session 2",
    ylabel="Mean rating (slider.response)",
    title="Participant means: Session 1 vs Session 2 ratings",
    save_name="paired_ratings_session1_vs_session2.png"
)
bar_sem_plot(
    means=[m_s1, m_s2],
    sems=[sem_s1, sem_s2],
    labels=["Session 1", "Session 2"],
    ylabel="Mean rating (slider.response)",
    title="Group mean ratings (Session 1 vs Session 2)",
    save_name="group_bar_ratings_session1_vs_session2.png"
)

paired_lines_plot(
    subj["mean_rt_s1"], subj["mean_rt_s2"],
    "Session 1", "Session 2",
    ylabel="Mean RT (slider.rt, seconds)",
    title="Participant means: Session 1 vs Session 2 RT",
    save_name="paired_rt_session1_vs_session2.png"
)
bar_sem_plot(
    means=[m_rt1, m_rt2],
    sems=[sem_rt1, sem_rt2],
    labels=["Session 1", "Session 2"],
    ylabel="Mean RT (slider.rt, seconds)",
    title="Group mean RT (Session 1 vs Session 2)",
    save_name="group_bar_rt_session1_vs_session2.png"
)

print(f"\nAll plots saved to: {OUTPUT_DIR}")


Found 44 CSVs total.
Identified 22 participant IDs with at least one session file.

Parsed trials: 14527 rows
      pid  session image_id  \
0  pid001        1     2345   
1  pid001        1     2071   
2  pid001        1     9160   
3  pid001        1     2100   
4  pid001        1     8030   

                                           stim_path resolution     rating  \
0  /Users/shamslab/Downloads/Resolution Study/sti...       high  69.833333   
1  /Users/shamslab/Downloads/Resolution Study/sti...       high  12.166667   
2  /Users/shamslab/Downloads/Resolution Study/sti...       high -18.666667   
3  /Users/shamslab/Downloads/Resolution Study/sti...        low -33.833333   
4  /Users/shamslab/Downloads/Resolution Study/sti...        low  -3.000000   

         rt  
0  4.306312  
1  3.714058  
2  3.419849  
3  2.901527  
4  4.183568  

Paired images across sessions: 6278 rows

Participant-level summary (first rows):
      pid  n_images  mean_rating_high  mean_rating_low  \
0  pid001

In [17]:
# =====================================================================
# CROSSED PARTICIPANT × STIMULUS LINEAR MIXED-EFFECTS MODEL
# =====================================================================
#
# PURPOSE
# -------
# Robustness analysis corresponding to the participant-aggregated
# 2 × 2 repeated-measures ANOVA.
#
# Fixed effects:
#     Valence:     positive vs. negative
#     Resolution:  high vs. low
#     Interaction: Valence × Resolution
#
# Crossed random effects:
#     (1 | participant)
#     (1 | stimulus)
#
# Model:
#     Enjoyment ~ Valence * Resolution
#                 + (1 | Participant)
#                 + (1 | Stimulus)
#
# IMPORTANT:
# Run this cell AFTER the main analysis cell in the notebook.
# The main cell creates `anov_long`, which contains the cleaned
# participant × stimulus × resolution observations.
#
# The current notebook reads the original data from:
#
# C:\Users\pkell\OneDrive\Documents\Grad School\Shams Lab\
# Resolution Study\data v1
#
# =====================================================================


# ---------------------------------------------------------------------
# 1. IMPORTS
# ---------------------------------------------------------------------

import os
import warnings
import numpy as np
import pandas as pd

from scipy import stats
import statsmodels
import statsmodels.formula.api as smf


# ---------------------------------------------------------------------
# 2. CHECK THAT THE MAIN ANALYSIS CELL HAS BEEN RUN
# ---------------------------------------------------------------------

if "anov_long" not in globals():
    raise RuntimeError(
        "\n`anov_long` was not found.\n\n"
        "Please run the main Resolution Study analysis cell first. "
        "That cell reads and cleans the raw data and creates the "
        "`anov_long` dataframe used here."
    )


# ---------------------------------------------------------------------
# 3. COPY THE DATA USED BY THE EXISTING ANOVA
# ---------------------------------------------------------------------

lmm_df = anov_long[
    [
        "pid",
        "IAPS",
        "valence",
        "resolution",
        "enjoy"
    ]
].copy()


# ---------------------------------------------------------------------
# 4. BASIC CLEANING
# ---------------------------------------------------------------------

# Keep only the two experimental valence categories
lmm_df = lmm_df[
    lmm_df["valence"].isin(["pos", "neg"])
].copy()

# Keep only high- and low-resolution observations
lmm_df = lmm_df[
    lmm_df["resolution"].isin(["high", "low"])
].copy()

# Ensure enjoyment is numeric
lmm_df["enjoy"] = pd.to_numeric(
    lmm_df["enjoy"],
    errors="coerce"
)

# Remove observations missing any variable required by the model
lmm_df = lmm_df.dropna(
    subset=[
        "pid",
        "IAPS",
        "valence",
        "resolution",
        "enjoy"
    ]
).copy()


# Treat participant and stimulus identifiers as categorical strings
lmm_df["pid"] = lmm_df["pid"].astype(str)
lmm_df["IAPS"] = lmm_df["IAPS"].astype(str)


# ---------------------------------------------------------------------
# 5. EFFECT CODE THE FIXED FACTORS
# ---------------------------------------------------------------------
#
# Coding:
#
# Valence
#     negative = -0.5
#     positive = +0.5
#
# Resolution
#     low  = -0.5
#     high = +0.5
#
# Advantages of this coding:
#
#   Intercept:
#       grand mean across all four conditions
#
#   valence_c:
#       positive − negative difference averaged across resolution
#
#   resolution_c:
#       high − low difference averaged across valence
#
#   valence_c:resolution_c:
#       difference in the resolution effect between
#       positive and negative stimuli
#
#       (PH − PL) − (NH − NL)
#
# ---------------------------------------------------------------------

lmm_df["valence_c"] = lmm_df["valence"].map(
    {
        "neg": -0.5,
        "pos":  0.5
    }
).astype(float)

lmm_df["resolution_c"] = lmm_df["resolution"].map(
    {
        "low":  -0.5,
        "high":  0.5
    }
).astype(float)


# ---------------------------------------------------------------------
# 6. CREATE A SINGLE TOP-LEVEL GROUP
# ---------------------------------------------------------------------
#
# statsmodels MixedLM is fundamentally organized around one grouping
# variable. To obtain genuinely CROSSED participant and stimulus
# random intercepts, we use one constant top-level group and specify
# BOTH participant and stimulus as variance components.
#
# This implements:
#
#     (1 | participant) + (1 | stimulus)
#
# rather than treating stimuli as nested within participants.
#
# ---------------------------------------------------------------------

lmm_df["_all"] = "all"


# ---------------------------------------------------------------------
# 7. REPORT THE DATA ENTERING THE MODEL
# ---------------------------------------------------------------------

n_obs = len(lmm_df)
n_participants = lmm_df["pid"].nunique()
n_stimuli = lmm_df["IAPS"].nunique()

print("\n" + "=" * 78)
print("CROSSED PARTICIPANT × STIMULUS MIXED-EFFECTS ANALYSIS")
print("=" * 78)

print(f"\nObservations entering model: {n_obs}")
print(f"Unique participants:         {n_participants}")
print(f"Unique stimuli:              {n_stimuli}")


print("\nObservations by condition:")

condition_counts = (
    lmm_df
    .groupby(
        ["valence", "resolution"],
        observed=True
    )
    .size()
    .reset_index(name="n_observations")
)

print(
    condition_counts.to_string(index=False)
)


# ---------------------------------------------------------------------
# 8. CONDITION DESCRIPTIVES
# ---------------------------------------------------------------------
#
# These are raw observation-level descriptives for reference.
# They are NOT substitutes for the model estimates.
#
# ---------------------------------------------------------------------

condition_descriptives = (
    lmm_df
    .groupby(
        ["valence", "resolution"],
        observed=True
    )
    .agg(
        mean_enjoy=("enjoy", "mean"),
        sd_enjoy=("enjoy", "std"),
        n_observations=("enjoy", "size"),
        n_participants=("pid", "nunique"),
        n_stimuli=("IAPS", "nunique"),
    )
    .reset_index()
)

print("\n" + "-" * 78)
print("RAW CONDITION DESCRIPTIVES")
print("-" * 78)

print(
    condition_descriptives.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ---------------------------------------------------------------------
# 9. SPECIFY THE CROSSED MIXED-EFFECTS MODEL
# ---------------------------------------------------------------------
#
# Fixed:
#     valence_c * resolution_c
#
# Random:
#     participant-specific intercept
#     stimulus-specific intercept
#
# use_sparse=True is useful because the participant/stimulus
# random-effect design matrices are large but sparse.
#
# ---------------------------------------------------------------------

print("\n" + "=" * 78)
print("FITTING CROSSED RANDOM-INTERCEPT MODEL")
print("=" * 78)

print(
    "\nModel:\n"
    "Enjoyment ~ Valence × Resolution "
    "+ (1 | Participant) + (1 | Stimulus)\n"
)

model = smf.mixedlm(
    formula="enjoy ~ valence_c * resolution_c",
    data=lmm_df,

    # One constant top-level group permits both participant
    # and stimulus to be represented as crossed variance components
    groups=lmm_df["_all"],

    # No additional top-level random intercept
    re_formula="0",

    # Crossed random intercepts
    vc_formula={
        "Participant": "0 + C(pid)",
        "Stimulus": "0 + C(IAPS)",
    },

    use_sparse=True
)


# ---------------------------------------------------------------------
# 10. FIT MODEL USING MAXIMUM LIKELIHOOD
# ---------------------------------------------------------------------

with warnings.catch_warnings(record=True) as caught_warnings:

    warnings.simplefilter("always")

    lmm_result = model.fit(
        reml=False,
        method="lbfgs",
        maxiter=2000,
        disp=False
    )


# ---------------------------------------------------------------------
# 11. MODEL CONVERGENCE
# ---------------------------------------------------------------------

print("\nConverged:", lmm_result.converged)

if not lmm_result.converged:
    print(
        "\nWARNING: The model did not report convergence.\n"
        "Do not interpret the fixed effects until the convergence "
        "issue has been resolved."
    )

if len(caught_warnings) > 0:

    print("\nModel-fitting warnings:")

    for w in caught_warnings:
        print(" -", str(w.message))


# ---------------------------------------------------------------------
# 12. FULL STATSMODELS SUMMARY
# ---------------------------------------------------------------------

print("\n" + "=" * 78)
print("FULL MIXED-MODEL SUMMARY")
print("=" * 78)

print(lmm_result.summary())


# ---------------------------------------------------------------------
# 13. EXTRACT FIXED-EFFECT RESULTS
# ---------------------------------------------------------------------

fixed_names = list(
    lmm_result.fe_params.index
)

fixed_beta = lmm_result.fe_params

fixed_se = lmm_result.bse_fe

fixed_z = fixed_beta / fixed_se

fixed_p = pd.Series(
    2.0 * stats.norm.sf(np.abs(fixed_z)),
    index=fixed_names
)

fixed_ci_low = fixed_beta - 1.96 * fixed_se
fixed_ci_high = fixed_beta + 1.96 * fixed_se


fixed_effects = pd.DataFrame(
    {
        "term": fixed_names,
        "b": fixed_beta.values,
        "SE": fixed_se.values,
        "z": fixed_z.values,
        "p": fixed_p.values,
        "CI95_low": fixed_ci_low.values,
        "CI95_high": fixed_ci_high.values,
    }
)


# Replace computer-oriented labels with manuscript-friendly labels

pretty_names = {
    "Intercept":
        "Grand mean",

    "valence_c":
        "Valence: positive − negative",

    "resolution_c":
        "Resolution: high − low",

    "valence_c:resolution_c":
        "Valence × Resolution interaction"
}

fixed_effects["effect"] = (
    fixed_effects["term"]
    .map(pretty_names)
    .fillna(fixed_effects["term"])
)


fixed_effects = fixed_effects[
    [
        "effect",
        "term",
        "b",
        "SE",
        "z",
        "p",
        "CI95_low",
        "CI95_high",
    ]
]


print("\n" + "=" * 78)
print("FIXED EFFECTS")
print("=" * 78)

print(
    fixed_effects.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ---------------------------------------------------------------------
# 14. FUNCTION FOR WALD CONTRASTS
# ---------------------------------------------------------------------
#
# This allows us to directly estimate:
#
#   Positive stimuli: PH − PL
#   Negative stimuli: NH − NL
#
# from the fitted mixed model.
#
# ---------------------------------------------------------------------

fe_cov = (
    lmm_result
    .cov_params()
    .loc[fixed_names, fixed_names]
)


def mixed_model_contrast(name, weights):

    """
    Calculate a linear contrast of the fixed effects.

    Parameters
    ----------
    name : str
        Human-readable contrast name.

    weights : dict
        Dictionary mapping fixed-effect parameter names
        to contrast weights.

    Returns
    -------
    dict
        Estimate, SE, z, p, and 95% CI.
    """

    L = np.array(
        [
            weights.get(term, 0.0)
            for term in fixed_names
        ],
        dtype=float
    )

    beta = fixed_beta.to_numpy(dtype=float)

    cov = fe_cov.to_numpy(dtype=float)

    estimate = float(
        L @ beta
    )

    variance = float(
        L @ cov @ L
    )

    se = float(
        np.sqrt(variance)
    )

    z = (
        estimate / se
        if se > 0
        else np.nan
    )

    p = (
        2.0 * stats.norm.sf(abs(z))
        if np.isfinite(z)
        else np.nan
    )

    ci_low = estimate - 1.96 * se
    ci_high = estimate + 1.96 * se

    return {
        "contrast": name,
        "estimate": estimate,
        "SE": se,
        "z": z,
        "p": p,
        "CI95_low": ci_low,
        "CI95_high": ci_high,
    }


# ---------------------------------------------------------------------
# 15. PLANNED SIMPLE-EFFECT CONTRASTS
# ---------------------------------------------------------------------
#
# With ±0.5 effect coding:
#
# Positive high − low:
#
#     b_resolution + 0.5*b_interaction
#
# Negative high − low:
#
#     b_resolution − 0.5*b_interaction
#
# Difference between those two effects:
#
#     b_interaction
#
# ---------------------------------------------------------------------

contrasts = []


# Positive stimuli: high − low
contrasts.append(
    mixed_model_contrast(
        name="Positive stimuli: high − low",
        weights={
            "resolution_c": 1.0,
            "valence_c:resolution_c": 0.5,
        }
    )
)


# Negative stimuli: high − low
contrasts.append(
    mixed_model_contrast(
        name="Negative stimuli: high − low",
        weights={
            "resolution_c": 1.0,
            "valence_c:resolution_c": -0.5,
        }
    )
)


# Direct interaction:
# positive resolution effect minus negative resolution effect
contrasts.append(
    mixed_model_contrast(
        name=(
            "Interaction contrast: "
            "(PH − PL) − (NH − NL)"
        ),
        weights={
            "valence_c:resolution_c": 1.0
        }
    )
)


contrast_df = pd.DataFrame(
    contrasts
)


print("\n" + "=" * 78)
print("MODEL-BASED PLANNED CONTRASTS")
print("=" * 78)

print(
    contrast_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ---------------------------------------------------------------------
# 16. RANDOM-EFFECT VARIANCE COMPONENTS
# ---------------------------------------------------------------------

print("\n" + "=" * 78)
print("RANDOM-EFFECT VARIANCE COMPONENTS")
print("=" * 78)

try:

    vc_names = list(
        lmm_result.model.exog_vc.names
    )

    vc_values = np.asarray(
        lmm_result.vcomp,
        dtype=float
    )

    variance_components = pd.DataFrame(
        {
            "component": vc_names,
            "variance": vc_values
        }
    )

    print(
        variance_components.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

except Exception as e:

    variance_components = pd.DataFrame()

    print(
        "Could not construct variance-component table:",
        repr(e)
    )


print(
    f"\nResidual variance: "
    f"{lmm_result.scale:.6f}"
)


# ---------------------------------------------------------------------
# 17. MODEL-FIT INFORMATION
# ---------------------------------------------------------------------

print("\n" + "=" * 78)
print("MODEL FIT")
print("=" * 78)

print(f"Log likelihood: {lmm_result.llf:.6f}")
print(f"AIC:            {lmm_result.aic:.6f}")
print(f"BIC:            {lmm_result.bic:.6f}")
print(f"Converged:      {lmm_result.converged}")


# ---------------------------------------------------------------------
# 18. SAVE OUTPUTS
# ---------------------------------------------------------------------
#
# Uses the same output directory as your main analysis if available.
# Otherwise creates:
#
#     DATA_DIR\resolution_analysis_outputs
#
# ---------------------------------------------------------------------

if "OUTPUT_DIR" not in globals() or OUTPUT_DIR is None:

    if "DATA_DIR" in globals():

        OUTPUT_DIR = os.path.join(
            DATA_DIR,
            "resolution_analysis_outputs"
        )

    else:

        OUTPUT_DIR = os.path.join(
            os.getcwd(),
            "resolution_analysis_outputs"
        )


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


fixed_effects_path = os.path.join(
    OUTPUT_DIR,
    "crossed_LMM_fixed_effects.csv"
)

contrasts_path = os.path.join(
    OUTPUT_DIR,
    "crossed_LMM_planned_contrasts.csv"
)

descriptives_path = os.path.join(
    OUTPUT_DIR,
    "crossed_LMM_condition_descriptives.csv"
)

variance_path = os.path.join(
    OUTPUT_DIR,
    "crossed_LMM_variance_components.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "crossed_LMM_full_summary.txt"
)


fixed_effects.to_csv(
    fixed_effects_path,
    index=False
)

contrast_df.to_csv(
    contrasts_path,
    index=False
)

condition_descriptives.to_csv(
    descriptives_path,
    index=False
)

if len(variance_components) > 0:

    variance_components.to_csv(
        variance_path,
        index=False
    )


with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "CROSSED PARTICIPANT × STIMULUS "
        "LINEAR MIXED-EFFECTS MODEL\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        f"Observations: {n_obs}\n"
    )

    f.write(
        f"Participants: {n_participants}\n"
    )

    f.write(
        f"Stimuli: {n_stimuli}\n\n"
    )

    f.write(
        "Model:\n"
        "Enjoyment ~ Valence × Resolution "
        "+ (1 | Participant) + (1 | Stimulus)\n\n"
    )

    f.write(
        "statsmodels version: "
        f"{statsmodels.__version__}\n\n"
    )

    f.write(
        str(lmm_result.summary())
    )

    f.write(
        "\n\nFIXED EFFECTS\n"
    )

    f.write(
        fixed_effects.to_string(
            index=False
        )
    )

    f.write(
        "\n\nPLANNED CONTRASTS\n"
    )

    f.write(
        contrast_df.to_string(
            index=False
        )
    )

    f.write(
        "\n"
    )


# ---------------------------------------------------------------------
# 19. FINAL OUTPUT SUMMARY
# ---------------------------------------------------------------------

print("\n" + "=" * 78)
print("FILES SAVED")
print("=" * 78)

print(f"\n{fixed_effects_path}")
print(f"{contrasts_path}")
print(f"{descriptives_path}")

if len(variance_components) > 0:
    print(f"{variance_path}")

print(f"{summary_path}")


print("\n" + "=" * 78)
print("KEY VALUES FOR THE MANUSCRIPT")
print("=" * 78)

print(
    "\nThe most important robustness test is:\n"
    "  Valence × Resolution interaction\n"
    "\nfollowed by the model-based contrasts:\n"
    "  1. Positive stimuli: high − low\n"
    "  2. Negative stimuli: high − low\n"
)

interaction_row = fixed_effects[
    fixed_effects["term"]
    == "valence_c:resolution_c"
]

if len(interaction_row) == 1:

    row = interaction_row.iloc[0]

    print(
        "Valence × Resolution:\n"
        f"  b = {row['b']:.3f}\n"
        f"  SE = {row['SE']:.3f}\n"
        f"  z = {row['z']:.3f}\n"
        f"  p = {row['p']:.6g}\n"
        f"  95% CI = "
        f"[{row['CI95_low']:.3f}, "
        f"{row['CI95_high']:.3f}]"
    )

print("\nAnalysis complete.")


CROSSED PARTICIPANT × STIMULUS MIXED-EFFECTS ANALYSIS

Observations entering model: 12428
Unique participants:         22
Unique stimuli:              394

Observations by condition:
valence resolution  n_observations
    neg       high            3165
    neg        low            3165
    pos       high            3049
    pos        low            3049

------------------------------------------------------------------------------
RAW CONDITION DESCRIPTIVES
------------------------------------------------------------------------------
valence resolution  mean_enjoy  sd_enjoy  n_observations  n_participants  n_stimuli
    neg       high    -41.6768   43.6004            3165              22        196
    neg        low    -41.2712   40.9354            3165              22        196
    pos       high     27.8947   38.1149            3049              22        198
    pos        low     21.6278   37.4173            3049              22        198

FITTING CROSSED RANDOM-INTERCEPT M